# Capítulo 17: Um Problema do Começo ao Fim

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

Um anúncio novo chega a uma plataforma de aluguel com a cidade, a área, os quartos, o condomínio e o IPTU, e falta o preço. A pergunta é quanto pedir, e o arquivo é `alugueis.csv`, com anúncios de cinco cidades brasileiras, do jeito que foram coletados. A seção 17.1 lê a tabela sem limpar nada e lista o que ela esconde: um traço no lugar do andar, linhas repetidas, valores extremos de área, condomínio e IPTU, e duas colunas que carregam o próprio aluguel, o total, que o soma, e o seguro-incêndio, que parece calculado sobre ele.

A seção 17.2 separa o teste antes de olhar a resposta e distingue as decisões que podem vir antes da divisão, como tirar as linhas repetidas e as colunas que contêm a resposta, das que são estimadas a partir das linhas e só podem ser tomadas no treino. É só no treino, depois da divisão, que a distribuição do aluguel é examinada, e dela sai a decisão de modelar o logaritmo do aluguel.

Um modelo só recebe números, e a conversão também aprende do dado: a lista das categorias, a média e o desvio de cada coluna numérica. A seção 17.3 monta essa conversão com o `ColumnTransformer`, uma transformação por tipo de coluna, e a põe no mesmo `Pipeline` que o modelo. Assim um `fit` ajusta os dois com as mesmas linhas, e um anúncio que chegar amanhã passa pela mesma conversão, com os números do treino. A seção mostra também o que acontece quando chega uma cidade que o treino não viu.

A escolha do modelo não pode gastar o teste. A seção 17.4 mostra por que o erro é medido na escala do logaritmo e compara a média, a regressão linear e o *k*-NN por validação cruzada, nos mesmos grupos, e lê a comparação grupo a grupo. A seção 17.5 escolhe com o `GridSearchCV` o número de vizinhos e, junto, uma decisão de preparo, a de passar ou não as colunas assimétricas pelo logaritmo. A seção 17.6 abre o teste uma vez, volta aos reais e relata o erro na unidade que responde à pergunta de quem vai anunciar, mostra onde o modelo erra mais, agrupando os anúncios pelo aluguel previsto e não pelo pedido, e fecha com o que o resultado não autoriza a concluir: o aluguel é o pedido, não o contratado, e associação não é efeito.

Ao final deste capítulo, você será capaz de:

- Definir a unidade, a resposta e os preditores de um problema de previsão pelo que se sabe no momento da previsão, e reconhecer as colunas que carregam a resposta
- Reconhecer valores extremos e zeros num preditor e escolher a escala, com o logaritmo, pelo efeito no gráfico e na distância entre anúncios
- Distinguir as decisões sobre o dado que podem vir antes da divisão em treino e teste das que são estimadas a partir das linhas, e explicar como linhas repetidas que atravessam a divisão inflam o acerto medido
- Montar o pré-processamento com `ColumnTransformer` e `Pipeline`, escolhendo uma transformação para cada tipo de coluna, e justificar por que tudo o que ele aprende é ajustado só no treino
- Comparar modelos por validação cruzada nos mesmos grupos, lendo a diferença grupo a grupo, e escolher a escala em que o erro é medido
- Ajustar com `GridSearchCV` hiperparâmetros do modelo e do pré-processamento ao mesmo tempo, e explicar por que o melhor erro de validação cruzada não é a estimativa final
- Relatar o erro no teste com a métrica que responde à pergunta de quem usa o modelo, medir onde o modelo erra mais agrupando pelo que se conhece na hora da previsão, e dizer o que o resultado não autoriza a concluir

## Seções

| Seção | Tópico |
|---|---|
| [17.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/01-o-problema-e-o-dado-cru.html) | O Problema e o Dado Cru |
| [17.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/02-separar-antes-de-olhar.html) | Separar antes de Olhar: Treino, Teste e Vazamento |
| [17.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/03-pre-processamento-como-parte-do-modelo.html) | Pré-processamento como Parte do Modelo: ColumnTransformer e Pipeline |
| [17.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/04-comparando-modelos-por-validacao-cruzada.html) | Comparando Modelos por Validação Cruzada |
| [17.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/05-ajuste-de-hiperparametros-com-gridsearchcv.html) | Ajuste de Hiperparâmetros com GridSearchCV |
| [17.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/06-reportar.html) | Reportar: a Métrica Certa, e o que o Resultado Não Diz |

## O Problema e o Dado Cru

Um anúncio novo chega a uma plataforma de aluguel. O dono informa a cidade, a área, quantos quartos, banheiros e vagas o imóvel tem, o andar, se aceita animal, se vem mobiliado, o condomínio e o IPTU. Falta o preço. Quanto ele deveria pedir por mês?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

### A pergunta

A unidade é **um anúncio**: cada linha da tabela é um imóvel oferecido para alugar. A resposta é **`aluguel`**, o valor mensal pedido, em reais. E os preditores só podem ser o que se sabe **no momento do anúncio**, antes de o preço existir. Além de `aluguel`, o arquivo traz estas colunas: `cidade`, `area_m2`, `quartos`, `banheiros`, `vagas`, `andar`, `aceita_animal`, `mobiliado`, `condominio`, `iptu`, `seguro_incendio` e `total`.

Pense antes de ler adiante: alguma delas só pode ter sido preenchida por alguém que já sabia o aluguel?

Duas: `seguro_incendio` e `total`. O total é uma soma que tem o aluguel como parcela, e o valor do seguro-incêndio, tudo indica, é calculado sobre o do aluguel. A seção 6.6 conferiu a soma de `total`, e a seção 17.2 mede quanto do aluguel há no seguro. Um modelo que as recebesse estaria lendo a resposta dentro da pergunta, e no anúncio novo, que ainda não tem preço, elas nem existiriam. Sobram as outras:

- `cidade`, `area_m2`, `quartos`, `banheiros`, `vagas` e `andar`, que descrevem o imóvel;
- `aceita_animal` e `mobiliado`, que descrevem a oferta;
- `condominio` e `iptu`, que vêm de fora do contrato: o condomínio, do rateio das despesas do prédio, e o IPTU, da prefeitura.

### O arquivo como ele vem

In [ ]:
cru = pd.read_csv("dados/alugueis.csv")
cru.shape

In [ ]:
cru.head(3).T

> **🔧 Função**
>
> **`pd.read_csv(caminho)`** — lê um arquivo CSV e devolve um `DataFrame`, com o tipo de cada coluna deduzido dos caracteres do arquivo.
>
> **`df.shape`** — atributo: o par (linhas, colunas).
>
> **`df.head(n)`** — as `n` primeiras linhas.
>
> **`df.T`** — atributo: a tabela transposta, com as linhas no lugar das colunas. Aqui, cada coluna do arquivo vira uma linha, e a tabela cabe na página.

São 10.692 anúncios e 13 colunas. Na tabela transposta, cada anúncio é uma coluna, e nos três primeiros `andar` traz números como qualquer outra coluna de contagem. O tipo com que o `pandas` leu cada coluna diz outra coisa:

In [ ]:
cru.dtypes

In [ ]:
cru.dtypes.value_counts()

> **🔧 Função**
>
> **`df.dtypes`** — atributo: o tipo de cada coluna.
>
> **`serie.value_counts()`** — quantas vezes cada valor distinto aparece na `Series`, do mais frequente ao menos frequente. Aqui, quantas colunas há de cada tipo.

Nove colunas chegaram como número (`int64`) e quatro como texto (`object`). Das quatro, três são texto de fato: `cidade`, `aceita_animal` e `mobiliado`. A quarta é `andar`, que deveria ser número. Contar os valores da coluna mostra por quê:

In [ ]:
cru["andar"].value_counts().head()

> **🔧 Função**
>
> **`df["coluna"]`** — a coluna com esse nome, como uma `Series`: os valores com o rótulo de cada linha.

O valor mais frequente de `andar` não é um número: é um traço, `"-"`, em 2.461 linhas. Basta um valor que não é número para a coluna inteira virar texto. Lendo de novo com o traço declarado como nulo:

In [ ]:
alugueis = pd.read_csv("dados/alugueis.csv", na_values=["-"])
alugueis.isna().sum()[lambda s: s > 0]

> **🔧 Função**
>
> **`pd.read_csv(caminho, na_values=marcadores)`** — `na_values` é uma lista de textos lidos como nulo, além dos que o `pandas` já reconhece. Aqui, `["-"]`.
>
> **`df.isna().sum()`** — `isna()` marca com `True` cada célula nula, e `.sum()` conta os `True` coluna a coluna. O `[lambda s: s > 0]` filtra e deixa só as colunas com algum nulo.

`andar` é a única coluna com nulo, e são os mesmos 2.461 traços. A seção 6.4 juntou indícios de que o traço marca uma casa, que não tem andar: nas linhas sem andar, o condomínio é quase sempre zero e a área mediana é maior. São indícios fortes, não prova, e a tabela tem linhas que não se encaixam neles:

In [ ]:
sem_andar = alugueis["andar"].isna()
com_condominio = alugueis["condominio"] > 0
print("sem andar e com condomínio:", (sem_andar & com_condominio).sum())

> **🔧 Função**
>
> **`serie > valor`** (e `==`, `<`) — compara cada elemento com `valor` e devolve uma `Series` de `True`/`False`, uma por linha. Uma `Series` assim é uma **máscara**.
>
> **`serie_a & serie_b`** — `True` só nas linhas em que as duas `Series` de `True`/`False` são `True`. Somado, conta essas linhas.

São 385 dos 2.461 anúncios sem andar com algum condomínio: uma casa em condomínio fechado, ou um apartamento cujo andar não foi informado. O arquivo não tem uma coluna que decida entre as duas leituras. O que fazer com esse nulo é decisão de preparo, e ela precisa servir aos dois casos.

### Linhas repetidas

In [ ]:
int(alugueis.duplicated().sum())

> **🔧 Função**
>
> **`df.duplicated()`** — `True` em cada linha idêntica, em todas as colunas, a uma linha que apareceu antes dela. Somado, conta as repetições.

São 358 linhas que repetem outra linha inteira, coluna por coluna. O arquivo não tem uma coluna que identifique o anúncio, então não há como saber se cada repetição é o mesmo anúncio publicado de novo ou dois imóveis iguais até o último real; a seção 6.4 mostrou três anúncios de Belo Horizonte nessa situação. Tirar ou manter as repetições muda o que o modelo aprende e, pior, muda quanto ele parece acertar: uma cópia no treino e outra no teste deixam o modelo ser julgado num anúncio que ele já viu, assunto da seção 17.2.

### Valores extremos

O resumo de `area_m2`, `condominio`, `iptu` e `andar` sai de um método só. Antes, uma função curta, `numero_br`, que só troca a pontuação de um número para o padrão brasileiro: o Python escreve `1,117,000.00`, com vírgula no milhar e ponto na decimal, e ela troca uma pela outra, sem mudar o valor. O argumento `casas` diz quantas casas decimais ficam.

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)
print(numero_br(1117000, 0))

In [ ]:
resumo = alugueis[["area_m2", "condominio", "iptu", "andar"]].describe()
resumo.style.set_uuid("resumo").format(lambda v: numero_br(v, 1)).format(
    lambda v: numero_br(v, 0), subset=pd.IndexSlice[["count"], :]
)

> **🔧 Função**
>
> **`df[["a", "b"]]`** — com uma lista de nomes dentro dos colchetes, uma tabela só com essas colunas.
>
> **`df.describe()`** — por coluna numérica: contagem de não nulos (`count`), média, desvio padrão, mínimo, os quartis `25%`, `50%` (a mediana) e `75%`, e o máximo.
>
> **`df.style.format(funcao, subset)`** — escreve cada célula com `funcao`, sem mudar o número guardado; com `subset`, só no recorte indicado. Aqui, `pd.IndexSlice[["count"], :]` recorta a linha `count` em todas as colunas, escrita sem casa decimal. **`.set_uuid(nome)`** fixa o identificador que o HTML da tabela usa.

A mediana de `area_m2` é 90 m² e o máximo é 46.335 m². A de `condominio` é R$ 560, e o máximo, R$ 1.117.000. O IPTU vai de uma mediana de R$ 125 a um máximo de R$ 313.700, e `andar`, de uma mediana de 5 a um máximo de 301. A contagem de `andar` é menor que a das outras porque os 2.461 nulos ficam de fora.

Um máximo sozinho não diz se ele é um ponto isolado ou o topo de uma cauda que sobe devagar. Os três maiores valores de cada coluna, e o maior dividido pelo segundo, respondem isso:

In [ ]:
colunas = ["area_m2", "condominio", "iptu", "andar"]
linhas = []
for coluna in colunas:
    primeiro, segundo, terceiro = alugueis[coluna].nlargest(3)
    linhas.append([primeiro, segundo, terceiro, primeiro / segundo,
                   primeiro / alugueis[coluna].median()])
extremos = pd.DataFrame(linhas, index=colunas,
                        columns=["maior", "2º", "3º", "÷ 2º", "÷ mediana"])
extremos.style.set_uuid("extremos").format(lambda v: numero_br(v, 0)).format(
    lambda v: numero_br(v, 1), subset=["÷ 2º"]
).set_table_styles(
    [{"selector": "th, td", "props": "padding: 0.15em 0.3em; font-size: 0.8em"}]
)

> **🔧 Função**
>
> **`serie.nlargest(n)`** — os `n` maiores valores da coluna, do maior para o menor. Com `n = 3`, os três se desempacotam em `primeiro, segundo, terceiro`.
>
> **`serie.median()`** — a mediana da coluna, ignorando os nulos.
>
> **`pd.DataFrame(linhas, index, columns)`** — monta uma tabela a partir de uma lista de linhas; `index` dá o nome de cada linha e `columns`, o de cada coluna.
>
> **`.set_table_styles(regras)`** — acrescenta regras de CSS à tabela, aqui uma letra menor para ela caber numa tela estreita. Os dois `.format` em sequência valem cada um no seu recorte: o segundo reescreve só a coluna `÷ 2º`.

A coluna `÷ 2º` é o maior dividido pelo segundo, e `÷ mediana`, o maior dividido pela mediana. Cada coluna tem o seu tipo de extremo. Em `iptu`, o maior valor é 11,2 vezes o segundo: um ponto isolado. Em `andar`, 301 é 5,9 vezes o segundo, 51, e um prédio de 301 andares parece erro de digitação. Em `area_m2` não há um só gigante, e sim três, de 46.335, 24.606 e 12.732 m², e o maior dá 515 vezes a mediana.

Em `condominio`, o maior valor aparece duas vezes, e a razão entre o primeiro e o segundo é 1,0. As duas linhas do topo são o mesmo anúncio?

In [ ]:
no_topo = alugueis["condominio"] == alugueis["condominio"].max()
print("linhas no topo:", no_topo.sum())
print("repetidas entre elas:", alugueis[no_topo].duplicated().sum())

> **🔧 Função**
>
> **`serie.max()`** e **`serie.min()`** — o maior e o menor valor da coluna.
>
> **`df[mascara]`** — só as linhas em que `mascara` é `True`.

Sim: uma das duas linhas repete a outra em todas as colunas, e é uma das 358 repetidas. Sem a linha repetida, o segundo maior condomínio passa a ser o de R$ 220.000, e o maior vale 5,1 vezes o segundo (1.117.000 ÷ 220.000). A repetição disfarçava um ponto isolado.

Nenhum desses extremos sai da tabela. Só pelo arquivo, não há como confirmar qual valor é erro, e tirar os que parecem errados muda a população de anúncios que o modelo vai atender: o anúncio novo pode trazer o mesmo erro de digitação. Para área, condomínio e IPTU, o que reduz o peso dos extremos é mudar a escala da coluna, e a figura adiante mostra o que isso faz. Antes, o outro lado dessas colunas, o mínimo:

In [ ]:
zero_condominio = alugueis["condominio"] == 0
print("condomínio zero:", numero_br(zero_condominio.sum(), 0))
print("IPTU zero:", numero_br((alugueis["iptu"] == 0).sum(), 0))
print("com andar e condomínio zero:", (zero_condominio & ~sem_andar).sum())

> **🔧 Função**
>
> **`~serie`** — troca `True` por `False` e vice-versa. Aqui, `~sem_andar` marca as linhas que têm andar.

São 2.373 anúncios com condomínio zero e 1.596 com IPTU zero. Parte desses zeros é plausível, como a casa sem condomínio, mas 297 anúncios têm andar informado (o que sugere um apartamento) e condomínio zero. Um zero pode ser o anunciante que deixou o campo em branco, a mesma dúvida do traço em `andar`. E o zero pesa na escolha da escala, porque o logaritmo de zero não existe.

### O que um eixo linear esconde

Um histograma de `area_m2` com o eixo em metros quadrados precisa ir de 11 a 46.335 para caber tudo. O que sobra, então, para os anúncios comuns? Antes de ver a figura, tente adivinhar quantas das 50 barras terão algum anúncio.

In [ ]:
# Figura: Área dos 10.692 anúncios, em dois histogramas de 50 barras. Em cima, eixo horizontal linear, de 11 a 46.335 m². Embaixo, eixo horizontal em escala logarítmica, com as barras igualmente largas nessa escala: a distribuição aparece. Nos dois painéis, cada triângulo cinza marca um dos três imóveis acima de 10 mil m², cujas barras são baixas demais para se ver.
fig, (ax_linear, ax_log) = plt.subplots(2, 1, figsize=(6.4, 6.4))
gigantes = alugueis.loc[alugueis["area_m2"] > 10_000, "area_m2"]
em_br = lambda v, pos: numero_br(v, 0)

ax_linear.hist(alugueis["area_m2"], bins=50,
               color="C0", edgecolor="white", linewidth=0.3)
ax_linear.set_xlim(0, alugueis["area_m2"].max() * 1.02)
ax_linear.set_title("eixo linear")
ax_linear.set_xlabel("área (m²)")

menor, maior = alugueis["area_m2"].min(), alugueis["area_m2"].max()
bordas_log = np.geomspace(menor, maior, 51)
ax_log.hist(alugueis["area_m2"], bins=bordas_log,
            color="C0", edgecolor="white", linewidth=0.3)
ax_log.set_xscale("log")
ax_log.set_xlim(10, maior * 1.5)
ax_log.set_title("eixo logarítmico")
ax_log.set_xlabel("área (m², escala log)")

for ax in (ax_linear, ax_log):
    topo = ax.get_ylim()[1]
    ax.scatter(gigantes, np.full(len(gigantes), 0.06 * topo), marker="v",
               s=40, color="0.45", zorder=3, label="_nolegend_")
    ax.xaxis.set_major_formatter(em_br)
    ax.yaxis.set_major_formatter(em_br)
    ax.set_ylabel("anúncios")

plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`plt.subplots(2, 1, figsize)`** — uma figura com duas linhas e uma coluna de painéis, um em cima do outro; devolve a figura e os dois eixos. `figsize` é (largura, altura) em polegadas.
>
> **`df.loc[linhas, coluna]`** — seleciona pelas linhas em que a condição é `True` e pelo nome da coluna.
>
> **`ax.hist(x, bins, color, edgecolor, linewidth)`** — histograma de `x`. Com um inteiro, `bins` é o número de barras de mesma largura; com um array, são as bordas das barras. `color` é a cor das barras (`"C0"`, o azul do ciclo), `edgecolor` e `linewidth`, a cor e a espessura do contorno.
>
> **`np.geomspace(inicio, fim, n)`** — `n` números de `inicio` a `fim` em progressão geométrica: a razão entre vizinhos é constante, e as barras ficam da mesma largura num eixo logarítmico.
>
> **`ax.set_xscale("log")`** — põe o eixo horizontal em escala logarítmica, em que cada multiplicação por 10 ocupa o mesmo espaço.
>
> **`ax.set_xlim(inicio, fim)`** e **`ax.get_ylim()`** — fixam o intervalo do eixo horizontal e leem o do vertical.
>
> **`ax.set_title(texto)`**, **`ax.set_xlabel(texto)`** e **`ax.set_ylabel(texto)`** — o título do painel e os rótulos dos eixos.
>
> **`ax.scatter(x, y, marker, s, color, zorder, label)`** — um ponto em cada par (`x`, `y`); `marker="v"` desenha um triângulo e `s` é o tamanho. `color="0.45"` é um cinza, `zorder=3` desenha os triângulos por cima das barras e `label="_nolegend_"` os deixa fora de qualquer legenda. Aqui, `y` é um array de `np.full(n, valor)`, `n` cópias do mesmo número, para os três triângulos ficarem na mesma altura.
>
> **`ax.xaxis.set_major_formatter(funcao)`** — escreve os rótulos do eixo com `funcao(valor, posicao)`; aqui, `numero_br`, com ponto de milhar.
>
> **`plt.tight_layout()`** e **`plt.show()`** — ajustam os espaços entre os painéis para os rótulos não se sobreporem, e exibem a figura.

As barras se contam com a mesma conta que o histograma faz antes de desenhar:

In [ ]:
contagens, bordas = np.histogram(alugueis["area_m2"], bins=50)
print("primeira barra (m²):", numero_br(bordas[0], 0), "a", numero_br(bordas[1], 0))
print("anúncios nela:", numero_br(contagens[0], 0), "de", numero_br(len(alugueis), 0))
print(f"fração: {numero_br(100 * contagens[0] / len(alugueis), 1)}%")
print("barras com anúncio:", (contagens > 0).sum(), "de 50")
print("acima de 10 mil m²:", (alugueis["area_m2"] > 10_000).sum())
print("quarto maior (m²):", numero_br(alugueis["area_m2"].nlargest(4).iloc[3], 0))

contagens_log, _ = np.histogram(alugueis["area_m2"], bins=bordas_log)
pico = contagens_log.argmax()
print("pico no eixo log (m²):", numero_br(bordas_log[pico], 0), "a",
      numero_br(bordas_log[pico + 1], 0))
print("anúncios no pico:", numero_br(contagens_log[pico], 0))

> **🔧 Função**
>
> **`np.histogram(x, bins)`** — a conta que `ax.hist` faz antes de desenhar, sem desenhar: devolve a contagem de cada barra e as bordas delas.
>
> **`serie.iloc[i]`** — o valor na posição `i`, contada a partir de 0. Aqui, `.nlargest(4).iloc[3]` é o quarto maior.
>
> **`array.argmax()`** — a posição do maior valor do array.

No eixo linear, cada barra tem 926 m² de largura (937 − 11), e a primeira, de 11 a 937 m², leva 10.669 dos 10.692 anúncios (99,8%). Só 6 das 50 barras têm algum anúncio, e as outras cinco são quase invisíveis ao lado da primeira. O eixo de cima tem o comprimento que o maior dos imóveis marcados pelos triângulos pede, e todo o resto se espreme na primeira barra. No eixo logarítmico, a mesma coluna vira um morro, com a barra mais alta entre 69 e 82 m² (1.133 anúncios), e os três imóveis acima de 10 mil m² continuam lá, isolados na ponta: abaixo deles, o maior imóvel tem 2.000 m², e eles não apagam o resto da distribuição.

A leitura vale para o modelo tanto quanto para o gráfico. O *k*-NN da seção 7.3 escolhe os vizinhos de um anúncio pela distância euclidiana: eleva ao quadrado a diferença em cada coluna, soma os quadrados e tira a raiz. Compare o imóvel de 46.335 m² com um imóvel mediano, coluna a coluna:

In [ ]:
contagens_imovel = ["area_m2", "quartos", "banheiros", "vagas"]
gigante = alugueis.loc[alugueis["area_m2"].idxmax(), contagens_imovel].astype(float)
mediano = alugueis[contagens_imovel].median()
diferenca = (gigante - mediano).abs()
print("área ÷ quartos:", numero_br(diferenca["area_m2"] / diferenca["quartos"], 1))
comparacao = pd.DataFrame(
    {"gigante": gigante, "mediano": mediano, "diferença": diferenca}
).T
comparacao.style.set_uuid("gigante").format(lambda v: numero_br(v, 0)).set_table_styles(
    [{"selector": "th, td", "props": "padding: 0.15em 0.3em; font-size: 0.8em"}]
)

> **🔧 Função**
>
> **`serie.idxmax()`** — o rótulo da linha em que está o maior valor.
>
> **`df.median()`** — a mediana de cada coluna. **`serie.abs()`** — o valor absoluto de cada elemento.
>
> **`serie.astype(float)`** — converte os valores para número de ponto flutuante.
>
> **`pd.DataFrame(dicionario)`** — com um dicionário, cada chave vira o nome de uma coluna e cada `Series`, os valores dela. O `.T` põe as três `Series` como linhas.

A diferença de área, 46.245 m², é 23.122,5 vezes a diferença de quartos, 2; os 6 banheiros e as 4 vagas de diferença também somem ao lado dela, e elevar ao quadrado aumenta ainda mais o peso da área. Os dois imóveis diferem em tudo, mas a distância só enxerga a área. Nenhuma linha precisa sair da tabela para isso mudar; o que muda é a escala. A conta abaixo mede a fatia da área na soma dos quadrados das diferenças, em três escalas: a do arquivo; cada coluna padronizada como na seção 7.3 (a média subtraída não muda a diferença entre dois imóveis, então basta dividir pelo desvio padrão); e a área em escala logarítmica antes de padronizar.

In [ ]:
imoveis = alugueis[contagens_imovel].astype(float)
area_em_log = imoveis.assign(area_m2=np.log1p(imoveis["area_m2"]))
linha_gigante = alugueis["area_m2"].idxmax()

def fatia_da_area(tabela, padronizar):
    dif = tabela.loc[linha_gigante] - tabela.median()
    if padronizar:
        dif = dif / tabela.std()
    quadrados = dif ** 2
    return 100 * quadrados["area_m2"] / quadrados.sum()

print(f"sem escala: {numero_br(fatia_da_area(imoveis, False), 1)}%")
print(f"padronizada: {numero_br(fatia_da_area(imoveis, True), 1)}%")
print(f"área em log, padronizada: {numero_br(fatia_da_area(area_em_log, True), 1)}%")

> **🔧 Função**
>
> **`df.assign(coluna=valores)`** — uma cópia da tabela com a coluna trocada (ou criada) por `valores`; a original não muda.
>
> **`np.log1p(x)`** — o logaritmo de `1 + x`, que existe também quando `x` é zero.
>
> **`df.std()`** — o desvio padrão de cada coluna.

Padronizar quase não mexe: a área fica com 99,6% da soma. Com a área em escala logarítmica antes de padronizar, a fatia cai para 69,1%. A área continua sendo o maior termo, mas perde o monopólio da distância, e as outras colunas passam a pesar na escolha dos vizinhos.

### O que fica para as próximas seções

Quanto o dono do anúncio novo deveria pedir? Antes de um modelo responder, três decisões de preparo continuam abertas: o que fazer com as linhas repetidas, que a seção 17.2 decide, e, na 17.3, como preencher o nulo de `andar` e como mudar a escala de `area_m2`, `condominio` e `iptu`. A 17.2 também mede quanto de `aluguel` há em `seguro_incendio`, que, com `total`, já ficou fora dos preditores.

Falta olhar a distribuição de `aluguel`, e ela espera a seção 17.2, que antes separa uma parte dos anúncios para o teste: examinar a resposta na tabela inteira deixaria os anúncios de teste influírem nas decisões que depois eles vão julgar. Esta seção olhou os preditores na tabela inteira, e isso é aceitável por dois motivos. Nenhuma decisão tomada aqui usa `aluguel`: declarar o traço como nulo e tirar dos preditores as colunas que carregam a resposta são regras fixas, que valem igual para qualquer anúncio. E o que o preparo precisar estimar a partir dos dados, como o valor que preenche um nulo, a seção 17.3 ajusta só nos anúncios de treino.

## Separar antes de Olhar: Treino, Teste e Vazamento

Quanto o modelo vai errar num anúncio que ainda não existe? A única forma de medir isso antes de o anúncio chegar é guardar uma parte dos anúncios que já existem e fingir que eles são novos: nenhuma decisão sobre o modelo pode tê-los visto. A pergunta difícil é o que conta como "ver".

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

### Três tipos de decisão sobre o dado

Padronizar `area_m2` é subtrair uma média e dividir por um desvio padrão. Se os dois números forem calculados na tabela inteira, os anúncios de teste entraram na conta: quando chegar a vez deles, serão julgados por um modelo que já recebeu algo deles. Se os dois números forem calculados só nos anúncios de treino, o teste é padronizado com eles, como um anúncio novo seria.

Deixar o teste entrar assim é **vazamento**: uma decisão sobre o modelo tomada com informação que um anúncio novo não teria, seja a resposta, seja as linhas do teste. A padronização ajustada na tabela inteira vaza pelas linhas do teste, com a média e o desvio dos preditores dele, sem olhar a resposta; a escolha das colunas mais correlacionadas com a resposta, feita na tabela inteira, vaza pela própria resposta do teste. A seção 10.6 mediu os dois vazamentos na validação cruzada, e a seção 17.3 evita os dois pondo o preparo dentro do modelo.

A média e o desvio são decisões **estimadas**: saem de uma conta sobre as linhas, e mudam se as linhas mudarem. Também são estimadas o valor que preenche um nulo, o limiar de valor extremo definido por um quantil e a escolha das colunas mais correlacionadas com a resposta. Outras decisões olham uma linha de cada vez. Tirar dos preditores uma coluna calculada a partir da resposta, ou declarar que o traço de `andar` é nulo, vale igual para a tabela toda e para um anúncio que chegar amanhã.

Tirar as linhas que repetem outra linha inteira fica entre os dois casos. Não estima número nenhum, mas só se sabe que uma linha é repetida comparando-a com as outras.

> **🔷 Conceito**
>
> **Cada tipo de decisão tem o seu lugar em relação à divisão:**
>
> - a **regra por linha** decide cada linha pelos valores dela, e pode vir antes ou depois;
> - **tirar repetições** compara as linhas entre si e vem antes; a subseção seguinte mostra por quê;
> - a **decisão estimada** vem depois: é calculada só no treino e aplicada ao teste com os mesmos números.

Olhar também é uma decisão estimada, só que feita a olho. Examinar a distribuição de `aluguel` na tabela inteira e escolher uma transformação pelo que se viu deixa os anúncios de teste influírem nessa escolha. Por isso a resposta só é examinada depois da divisão, e só no treino.

### Linhas repetidas atravessam a divisão

O arquivo tem linhas que repetem outra linha inteira. Se ele for dividido assim, uma cópia pode cair no treino e a outra no teste. Os números impressos adiante usam a função `numero_br` da seção 17.1, que escreve um número com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

A divisão abaixo separa 20% dos anúncios para o teste, **sem** tirar as repetições antes:

In [ ]:
com_repeticao = pd.read_csv("dados/alugueis.csv", na_values=["-"])
treino_r, teste_r = train_test_split(
    com_repeticao, test_size=0.2, random_state=17,
    stratify=com_repeticao["cidade"],
)
print("linhas repetidas:", com_repeticao.duplicated().sum())
print("treino:", numero_br(len(treino_r), 0))
print("teste:", numero_br(len(teste_r), 0))

> **🔧 Função**
>
> **`train_test_split(tabela, test_size, random_state, stratify)`** — sorteia as linhas da tabela em duas partes e devolve primeiro a de treino, depois a de teste. `test_size=0.2` põe 20% das linhas no teste; `random_state=17` fixa o sorteio, e a divisão sai igual a cada execução; `stratify` é explicado adiante, em "A divisão".

São 358 linhas repetidas, e a divisão põe 8.553 anúncios no treino e 2.139 no teste. Quantos anúncios do teste têm uma cópia idêntica no treino? Um `merge` casa cada linha do teste com as linhas do treino que têm os mesmos valores em todas as colunas:

In [ ]:
cruzamento = teste_r.merge(treino_r.drop_duplicates(), how="left", indicator=True)
assert len(cruzamento) == len(teste_r)
tem_copia = (cruzamento["_merge"] == "both").to_numpy()
print("com cópia no treino:", tem_copia.sum(), "de", numero_br(len(teste_r), 0))

> **🔧 Função**
>
> **`df.merge(outra, how, indicator)`** — junta duas tabelas pelas colunas que elas têm em comum, aqui todas. Com `how="left"`, toda linha de `df` aparece no resultado, na mesma ordem, tenha par em `outra` ou não. Com `indicator=True`, a coluna `_merge` diz de onde veio cada linha: `"both"` quando houve par, `"left_only"` quando não houve.
>
> **`df.drop_duplicates()`** — a tabela sem as linhas que repetem uma linha anterior. Aqui, no treino, para que um anúncio com duas cópias no treino não apareça duas vezes no resultado do `merge`; o `assert` confere que o resultado tem uma linha por anúncio de teste.
>
> **`serie.to_numpy()`** — os valores da `Series` como um array do `numpy`, sem os rótulos das linhas.

São 91 dos 2.139 anúncios de teste com uma cópia exata no treino. Para um modelo que decora o treino, esses 91 não são anúncios novos. O *k*-NN da seção 7.3 com um vizinho só é esse modelo: prevê para cada anúncio o aluguel do anúncio de treino mais parecido. Antes de ver o resultado, pense: o que ele prevê para um anúncio de teste que tem uma cópia no treino?

O chunk abaixo usa as seis colunas numéricas sem nulo (`andar` fica de fora porque o *k*-NN não aceita nulo), sem padronizar: para uma cópia exata, a distância é zero em qualquer escala. Ele mede o erro absoluto, em reais, de cada anúncio de teste:

In [ ]:
numericas = ["area_m2", "quartos", "banheiros", "vagas", "condominio", "iptu"]
um_vizinho = KNeighborsRegressor(n_neighbors=1)
um_vizinho.fit(treino_r[numericas], treino_r["aluguel"])
erro = (um_vizinho.predict(teste_r[numericas]) - teste_r["aluguel"]).abs()

print("erro mediano, cópias: R$", numero_br(erro[tem_copia].median(), 0))
print("erro mediano, demais: R$", numero_br(erro[~tem_copia].median(), 0))
print("erro zero em todas as cópias:", (erro[tem_copia] == 0).all())

> **🔧 Função**
>
> **`KNeighborsRegressor(n_neighbors)`** — o *k*-NN para resposta numérica: prevê a média da resposta dos `n_neighbors` anúncios de treino mais próximos. Com `n_neighbors=1`, o aluguel do mais próximo.
>
> **`modelo.fit(X, y)`** e **`modelo.predict(X)`** — `fit` ajusta o modelo às colunas `X` e à resposta `y` do treino; `predict` devolve a previsão para cada linha de outra tabela com as mesmas colunas.
>
> **`serie.all()`** — `True` se todos os elementos de uma `Series` de `True`/`False` são `True`.

Nas cópias, o erro mediano é zero: o vizinho mais próximo está à distância zero, e é a cópia do anúncio no treino. Nos demais anúncios de teste, o erro mediano é de R$ 1.000. A figura compara o erro nos dois grupos de anúncios de teste:

In [ ]:
# Figura: Erro absoluto do *k*-NN com um vizinho nos anúncios de teste, em barras de R$ 250; a altura de cada barra é a fração do grupo. Erros acima de R$ 6.000 estão somados na última barra, e os dois painéis têm a mesma escala vertical. Em cima, os 91 anúncios com cópia no treino; embaixo, os 2.048 demais.
bordas_erro = np.arange(0, 6_001, 250)
em_br = lambda v, pos: numero_br(v, 0)
em_pct = lambda v, pos: numero_br(100 * v, 0) + "%"
grupos = [
    (erro[tem_copia], "C1", "com cópia no treino"),
    (erro[~tem_copia], "C0", "sem cópia no treino"),
]
fig, eixos = plt.subplots(2, 1, figsize=(5, 5.2), sharex=True, sharey=True)
for ax, (grupo, cor, nome) in zip(eixos, grupos):
    ax.hist(grupo.clip(upper=5_999), bins=bordas_erro,
            weights=np.full(len(grupo), 1 / len(grupo)),
            color=cor, edgecolor="white", linewidth=0.3)
    ax.set_title(f"{numero_br(len(grupo), 0)} anúncios {nome}")
    ax.set_ylabel("fração do grupo")
    ax.yaxis.set_major_formatter(em_pct)
eixos[1].set_xlim(0, 6_000)
eixos[1].xaxis.set_major_formatter(em_br)
eixos[1].set_xlabel("erro absoluto (R$)")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.arange(inicio, fim, passo)`** — os números de `inicio` até antes de `fim`, de `passo` em `passo`. Aqui, as bordas das barras, de R$ 250 em R$ 250.
>
> **`serie.clip(upper=valor)`** — troca todo valor acima de `valor` por `valor`; aqui, junta a cauda na última barra.
>
> **`ax.hist(x, bins, weights)`** — com `weights`, cada valor soma o seu peso à barra em vez de 1. Com peso 1/n para os n anúncios do grupo, a altura da barra é a fração do grupo, e grupos de tamanhos diferentes ficam comparáveis.
>
> **`plt.subplots(..., sharex=True, sharey=True)`** — faz os dois painéis dividirem os dois eixos, e a mesma altura quer dizer a mesma fração em cima e embaixo.

Nem todas as cópias acertam em cheio, e o booleano acima deu `False`: o treino tem anúncios iguais nas seis colunas numéricas e com aluguéis diferentes. Aí vários vizinhos empatam à distância zero, e qual deles ganha depende da ordem das linhas no treino e do algoritmo que o *k*-NN usa para buscar os vizinhos. Acertam qualquer que seja o desempate as cópias cujas seis colunas aparecem no treino com um aluguel só:

In [ ]:
distintos = treino_r[numericas + ["aluguel"]].drop_duplicates()
unicos = distintos[~distintos.duplicated(numericas, keep=False)]
certas = len(teste_r[tem_copia].merge(unicos[numericas], on=numericas))
print("acerto garantido:", certas, "de", tem_copia.sum())
print("mais da metade:", certas > tem_copia.sum() / 2)

> **🔧 Função**
>
> **`df.duplicated(colunas, keep=False)`** — compara as linhas só nas `colunas`; com `keep=False`, marca **todas** as linhas de um grupo repetido, e não só as que vêm depois da primeira. Aqui, as combinações das seis colunas que aparecem com mais de um aluguel.
>
> **`df.merge(outra, on=colunas)`** — casa as linhas só pelas `colunas`; sem `how`, fica só com as linhas que têm par nas duas tabelas.

São 71 das 91 cópias com acerto garantido, mais da metade: o erro mediano zero não depende do desempate.

O erro mediano zero nas cópias mostra só que o modelo guardou o treino, e nada sobre o que ele aprendeu de aluguel. Essa divisão serve para mostrar o problema e é descartada aqui.

A saída é tirar as repetições **antes** de dividir. Feito depois, cada metade tira só as repetições que tem dentro de si:

In [ ]:
alugueis = com_repeticao.drop_duplicates()
depois = len(treino_r.drop_duplicates()) + len(teste_r.drop_duplicates())
print("tirando antes de dividir:", numero_br(len(alugueis), 0))
print("tirando depois de dividir:", numero_br(depois, 0))

Antes de dividir, sobram 10.334 anúncios; depois, 10.414. A diferença, 80 (10.414 − 10.334), corresponde a linhas com cópia dos dois lados, que nenhuma das metades, olhada sozinha, reconhece como repetida. Continua valendo a ressalva da seção 6.4, que mostrou anúncios de Belo Horizonte idênticos até o último real: sem uma coluna que identifique o anúncio, não há como saber se uma repetição é o mesmo imóvel anunciado duas vezes ou dois imóveis iguais. Tirá-la supõe o primeiro caso.

### A divisão

As cinco cidades não têm o mesmo número de anúncios, e um sorteio simples pode deixar a fração de uma cidade no teste diferente da fração dela no treino. O argumento `stratify` sorteia dentro de cada cidade, separando 20% de cada uma. Antes de ver a tabela, pense: se uma cidade tem poucos anúncios, o que um sorteio simples pode fazer com a fração dela no teste? A tabela compara a fração de cada cidade na tabela inteira e no teste, sem e com `stratify`:

In [ ]:
treino_s, teste_s = train_test_split(alugueis, test_size=0.2, random_state=17)
treino, teste = train_test_split(
    alugueis, test_size=0.2, random_state=17, stratify=alugueis["cidade"]
)

fracao = lambda tabela: tabela["cidade"].value_counts(normalize=True)
proporcoes = pd.DataFrame(
    {"tabela": fracao(alugueis), "sem": fracao(teste_s), "com": fracao(teste)}
)
(
    proporcoes.style.set_uuid("proporcoes")
    .format(lambda v: numero_br(v, 3))
    .set_table_styles([{"selector": "th, td", "props": "padding: 0.15em 0.3em"}])
)

> **🔧 Função**
>
> **`serie.value_counts(normalize=True)`** — com `normalize=True`, a fração de cada valor em vez da contagem.

In [ ]:
diferenca_sem = (fracao(treino_s) - fracao(teste_s)).abs().max()
diferenca_com = (fracao(treino) - fracao(teste)).abs().max()
print("maior diferença treino × teste")
print("  sem stratify:", numero_br(diferenca_sem, 4))
print("  com stratify:", numero_br(diferenca_com, 4))
print("treino:", numero_br(len(treino), 0), "| teste:", numero_br(len(teste), 0))
menor = fracao(teste).index[-1]
print("teste em", menor + ":", (teste["cidade"] == menor).sum())

> **🔧 Função**
>
> **`serie.index`** — os rótulos das linhas da `Series`. Na saída de `value_counts`, os rótulos são os valores contados, do mais frequente ao menos frequente, e `.index[-1]` é o último, o menos frequente.

Sem `stratify`, a fração de uma cidade no treino e no teste chega a diferir em 0,0072; com ele, a maior diferença é de 0,0003. Com cinco cidades e milhares de anúncios, o sorteio simples já sai perto; a diferença cresce quando uma categoria é rara ou a tabela é pequena, e aí estratificar faz diferença. A estratificação é por cidade porque a seção 17.6 mede o erro também cidade por cidade, e cada cidade precisa estar no teste na mesma fração que tem no treino. Campinas, a cidade com a menor fração no teste, tem 165 anúncios nele, e é com eles que o erro de Campinas vai ser medido.

Os 20% são uma escolha comum, e não uma regra. O teste precisa ser grande o bastante para que o erro medido nele oscile pouco, e cada anúncio que vai para o teste é um a menos para ajustar o modelo. A divisão que fica é a estratificada: 8.267 anúncios de treino e 2.067 de teste.

A partir daqui, o `teste` fica guardado. Daqui até a seção 17.5, nenhuma decisão é tomada olhando para ele, e ele é aberto uma vez, na seção 17.6, para medir o erro do modelo escolhido.

### Colunas que contêm a resposta

Duas colunas só existem depois que o aluguel é fixado: `total` e `seguro_incendio`, que a seção 17.1 deixou fora dos preditores por isso. A exclusão vem do que as colunas são, não de uma conta sobre o dado. Para `total`, a relação é uma fórmula: a seção 6.6 conferiu que, na grande maioria das linhas, ele é exatamente a soma de aluguel, condomínio, IPTU e seguro-incêndio. Para o seguro, a relação não está escrita em lugar nenhum do arquivo. Se ele for calculado como uma fração do aluguel, a razão entre os dois deve ser quase a mesma em todos os anúncios. A medição abaixo só confirma a exclusão, e é feita no treino, porque usa `aluguel`, a resposta:

In [ ]:
razao = treino["seguro_incendio"] / treino["aluguel"]
fator = razao.median()
q05, q95 = razao.quantile(0.05), razao.quantile(0.95)
print("razão mediana:", numero_br(fator, 5))
print("90% entre", numero_br(q05, 4), "e", numero_br(q95, 4))

previsto = treino["seguro_incendio"] / fator
erro_relativo = (previsto - treino["aluguel"]).abs() / treino["aluguel"]
print(f"erro relativo mediano: {numero_br(100 * erro_relativo.median(), 2)}%")
print(f"a menos de 10%: {numero_br(100 * (erro_relativo < 0.10).mean(), 1)}%")

> **🔧 Função**
>
> **`serie.quantile(q)`** — o valor abaixo do qual fica a fração `q` dos dados. Os quantis 0,05 e 0,95 delimitam os 90% centrais.
>
> **`serie.mean()`** — a média. Numa `Series` de `True`/`False`, é a fração de `True`.

A razão mediana é 0,01313, e 90% dos anúncios de treino têm razão entre 0,0127 e 0,0160. É um "modelo" com um número só: dividir o seguro por 0,01313 devolve o aluguel com erro relativo mediano de 2,96%, e 67,6% dos anúncios ficam a menos de 10% do aluguel verdadeiro. Não é uma fórmula exata, já que os outros 32,4% (100 − 67,6) passam dos 10%, mas é perto o bastante para o seguro entregar boa parte do aluguel. Um modelo que recebesse o seguro poderia aprender essa divisão, e o erro medido no teste, que também tem o seguro preenchido, deixaria de dizer quanto ele erra num anúncio novo. No anúncio novo, que ainda não tem preço, o seguro nem existe.

In [ ]:
# Figura: Seguro-incêndio contra aluguel nos 8.267 anúncios de treino, com os dois eixos em escala logarítmica. Cada ponto azul é um anúncio. A reta laranja é seguro = 0,01313 × aluguel, com 0,01313 a razão mediana entre os dois no treino: a nuvem se alinha a ela.
fig, ax = plt.subplots(figsize=(6.4, 4.8))
ax.scatter(treino["aluguel"], treino["seguro_incendio"],
           s=6, alpha=0.15, color="C0", linewidths=0, label="anúncio de treino")
faixa = np.geomspace(treino["aluguel"].min(), treino["aluguel"].max(), 100)
ax.plot(faixa, fator * faixa, color="C1", linewidth=2,
        label=f"seguro = {numero_br(fator, 5)} × aluguel")
ax.set_xscale("log")
ax.set_yscale("log")
ax.xaxis.set_major_formatter(em_br)
ax.yaxis.set_major_formatter(em_br)
ax.set_xlabel("aluguel (R$, escala log)")
ax.set_ylabel("seguro-incêndio (R$, escala log)")
ax.set_xlim(400, 30_000)
ax.set_ylim(2, 500)
legenda = ax.legend(loc="upper left")
legenda.legend_handles[0].set_alpha(1)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.scatter(..., alpha, linewidths)`** — `alpha=0.15` deixa cada ponto quase transparente, e onde muitos se sobrepõem a cor fica mais forte; `linewidths=0` tira o contorno dos pontos.
>
> **`ax.plot(x, y, color, linewidth, label)`** — uma linha ligando os pontos (`x`, `y`); `label` é o texto dela na legenda.
>
> **`ax.set_yscale("log")`** — põe o eixo vertical em escala logarítmica. Com os dois eixos em log, uma relação `y = c × x` vira uma reta de inclinação 1, qualquer que seja `c`.
>
> **`ax.set_ylim(inicio, fim)`** — fixa o intervalo do eixo vertical.
>
> **`ax.legend(loc)`** — desenha a legenda com os `label` de cada elemento; `loc` escolhe o canto. Os pontos têm transparência (`alpha=0.15`), e `legenda.legend_handles[0].set_alpha(1)` desenha o ponto da legenda opaco, para ele se ver.

A decisão é tirar as duas colunas. Os preditores são os dez que a seção 17.1 listou:

In [ ]:
preditores = ["cidade", "area_m2", "quartos", "banheiros", "vagas", "andar",
              "aceita_animal", "mobiliado", "condominio", "iptu"]
print("total entre os preditores:", "total" in preditores)
print("seguro entre os preditores:", "seguro_incendio" in preditores)

### Agora, olhar o treino

Com o teste guardado, a distribuição de `aluguel` pode ser examinada, no treino. Uma pergunta antes: se a transformação fosse escolhida olhando o aluguel da tabela inteira, que anúncios estariam influindo numa decisão que o teste deveria julgar?

In [ ]:
aluguel = treino["aluguel"]
print("mínimo: R$", numero_br(aluguel.min(), 0))
print("mediana: R$", numero_br(aluguel.median(), 0))
print("média: R$", numero_br(aluguel.mean(), 0))
print("máximo: R$", numero_br(aluguel.max(), 0))
print("assimetria em reais:", numero_br(aluguel.skew(), 2))
print("assimetria do log:", numero_br(np.log(aluguel).skew(), 2))

> **🔧 Função**
>
> **`serie.skew()`** — a assimetria da coluna: perto de zero numa distribuição simétrica, positiva quando a cauda da direita é mais longa que a da esquerda.
>
> **`np.log(x)`** — o logaritmo natural de cada elemento. Só existe para valores positivos.

A média, R$ 3.960, fica acima da mediana, R$ 2.750, o sinal de uma cauda longa à direita. A assimetria em reais é 1,64, e o logaritmo do aluguel tem assimetria de 0,21, bem mais perto de uma distribuição simétrica. A figura mostra as duas escalas:

In [ ]:
# Figura: Aluguel dos 8.267 anúncios de treino, em dois histogramas de 30 barras. Em cima, eixo em reais: a cauda da direita é longa. Embaixo, eixo em escala logarítmica, com as barras igualmente largas nessa escala; a forma é a da distribuição de log(aluguel), e o morro fica quase simétrico.
fig, (ax_reais, ax_log) = plt.subplots(2, 1, figsize=(6.4, 6.4))

ax_reais.hist(aluguel, bins=30, color="C0", edgecolor="white", linewidth=0.3)
ax_reais.set_title("eixo em reais")
ax_reais.set_xlabel("aluguel (R$)")
ax_reais.set_xlim(0, 26_000)

bordas_log = np.geomspace(aluguel.min(), aluguel.max(), 31)
ax_log.hist(aluguel, bins=bordas_log, color="C0", edgecolor="white", linewidth=0.3)
ax_log.set_xscale("log")
ax_log.set_xlim(400, 30_000)
ax_log.set_title("eixo logarítmico")
ax_log.set_xlabel("aluguel (R$, escala log)")

for ax in (ax_reais, ax_log):
    ax.xaxis.set_major_formatter(em_br)
    ax.yaxis.set_major_formatter(em_br)
    ax.set_ylabel("anúncios")

plt.tight_layout()
plt.show()

A barra mais alta de cada histograma:

In [ ]:
contagens, bordas = np.histogram(aluguel, bins=30)
contagens_log, _ = np.histogram(aluguel, bins=bordas_log)
i, j = contagens.argmax(), contagens_log.argmax()
print("em reais: R$", numero_br(bordas[i], 0), "a", numero_br(bordas[i + 1], 0))
print("no log: R$", numero_br(bordas_log[j], 0), "a", numero_br(bordas_log[j + 1], 0))

No eixo em reais, a barra mais alta fica bem à esquerda, de R$ 1.268 a R$ 2.087, e a cauda se estende até o máximo de R$ 25.000. No eixo log, a barra mais alta vai de R$ 1.963 a R$ 2.244.

A decisão é modelar o logaritmo do aluguel. O menor aluguel do treino é positivo, e o logaritmo existe para todos. Se uma característica do imóvel multiplica o preço (10% a mais, digamos) em vez de somar um valor fixo em reais, na escala log esse efeito vira uma soma, que é a forma com que a regressão linear combina os preditores. E o erro passa a ser relativo. Errar R$ 200 num aluguel de R$ 2.000 ou R$ 2.000 num de R$ 20.000 é errar 10% nos dois casos, e as duas diferenças são iguais na escala log:

In [ ]:
print(numero_br(np.log(2_200) - np.log(2_000), 4))
print(numero_br(np.log(22_000) - np.log(20_000), 4))

As duas diferenças dão 0,0953, o logaritmo de 1,1. O preço dessa escolha é que a previsão sai em log, e dizer quanto o modelo erra em reais exige voltar da escala log, o que a seção 17.6 faz.

Com a decisão tomada, os preditores e a resposta do treino ficam assim:

In [ ]:
y_treino = np.log(treino["aluguel"])
X_treino = treino[preditores]
print("X_treino:", X_treino.shape)
print("y_treino:", y_treino.shape)

### O que fica decidido

Ao fim desta seção, o dado está neste estado, e é dele que o preparo e os modelos partem:

1. a tabela sem as linhas repetidas, com 10.334 anúncios;
2. a divisão estratificada por cidade, com `random_state=17`: 8.267 anúncios de treino e 2.067 de teste, guardados até a seção 17.6;
3. os dez `preditores`, sem `total` e sem `seguro_incendio`;
4. a resposta `y_treino = np.log(treino["aluguel"])`.

A seção 17.3 decide o que ainda falta: o nulo de `andar`, a escala das colunas assimétricas e a codificação de `cidade`, com tudo o que for estimado ajustado só no treino.

## Pré-processamento como Parte do Modelo: ColumnTransformer e Pipeline

Uma regressão linear ou um *k*-NN só recebem números. A cidade chega por extenso, `andar` chega vazio em parte dos anúncios, e a área, em metros quadrados, está numa escala muito diferente da de uma contagem de quartos. Alguém precisa converter tudo isso em números comparáveis, e algumas conversões usam números aprendidos do dado: uma média, um desvio, a lista das cidades. Aprendidos de quais anúncios? E como garantir que um anúncio que chegar amanhã passe exatamente pela mesma conversão?

In [ ]:
import textwrap
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

A função `numero_br` escreve os números com a pontuação brasileira. O ponto de partida é a tabela sem linhas repetidas, a divisão estratificada por cidade, os dez preditores e o logaritmo do aluguel como resposta.

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

In [ ]:
alugueis = pd.read_csv("dados/alugueis.csv", na_values=["-"]).drop_duplicates()
preditores = ["cidade", "area_m2", "quartos", "banheiros", "vagas", "andar",
              "aceita_animal", "mobiliado", "condominio", "iptu"]
treino, teste = train_test_split(alugueis, test_size=0.2, random_state=17,
                                 stratify=alugueis["cidade"])
X_treino, y_treino = treino[preditores], np.log(treino["aluguel"])
print("X_treino:", X_treino.shape)
print("andar nulo no treino:", numero_br(X_treino["andar"].isna().sum(), 0))
print("colunas com nulo:", list(X_treino.columns[X_treino.isna().any()]))

> **🔧 Função**
>
> **`df.isna().any()`** — para cada coluna, `True` se ela tem ao menos um nulo. Usado como filtro de `df.columns`, deixa só os nomes dessas colunas.

Daqui em diante, tudo o que aprende algum número do dado aprende só de `X_treino`. O `teste` continua guardado.

### Uma transformação por tipo de coluna

Os dez preditores não pedem todos a mesma conversão. Três são texto, três são valores em metros quadrados ou em reais com cauda longa, um tem nulo e três são contagens. Cada grupo ganha a sua transformação.

**As categóricas.** `cidade` tem cinco valores, e `aceita_animal` e `mobiliado`, dois cada (`"sim"` e `"não"`). Dar um número a cada cidade (1 para Belo Horizonte, 2 para Campinas, e assim por diante) inventaria uma ordem e uma distância que não existem: Campinas não fica "entre" Belo Horizonte e Porto Alegre. A codificação ***one-hot*** (em tradução livre, "um aceso") cria uma coluna por valor, com 1 na coluna da cidade do anúncio e 0 nas demais:

In [ ]:
codificador = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
cidades = codificador.fit_transform(X_treino[["cidade"]])
cidades.head(3).T.astype(int)

> **🔧 Função**
>
> **`OneHotEncoder(sparse_output)`** — a codificação one-hot: uma coluna de 0 e 1 para cada valor da coluna original. Com `sparse_output=False`, o resultado é uma tabela comum, e não o formato compacto que guarda só as posições dos 1.
>
> **`.set_output(transform="pandas")`** — faz o transformador devolver um `DataFrame`, com o nome de cada coluna, em vez de um array sem nomes.
>
> **`transformador.fit_transform(X)`** — `fit` aprende de `X` o que a transformação precisa (aqui, a lista das cidades), e `transform` aplica a transformação a `X`; `fit_transform` faz os dois de uma vez.

As cinco colunas saíram do `fit`, que guardou as cidades que apareceram no treino. Os três anúncios da tabela têm um 1 e quatro 0, e o treino inteiro também:

In [ ]:
print("um único 1 por anúncio:", bool((cidades.sum(axis=1) == 1).all()))

> **🔧 Função**
>
> **`df.sum(axis=1)`** — soma cada linha, atravessando as colunas; sem `axis=1`, a soma é de cada coluna.

**As assimétricas**, as três de cauda longa. `area_m2`, `condominio` e `iptu` têm a cauda longa que a seção 17.1 mostrou, e lá, entre o imóvel gigante e um mediano, a área em escala logarítmica perdeu o monopólio da distância. O logaritmo de zero não existe, e zero não é raro em duas delas:

In [ ]:
print("condomínio zero:", numero_br((X_treino["condominio"] == 0).sum(), 0))
print("IPTU zero:", numero_br((X_treino["iptu"] == 0).sum(), 0))

Por isso a transformação é `np.log1p`, o logaritmo natural de 1 + x, que leva o zero ao zero.

Depois dela vem a padronização: subtrair a média e dividir pelo desvio padrão, com a média e o desvio calculados no treino. Ela põe as colunas numa escala comum, a do desvio padrão. Sozinha, não basta. Para o par do imóvel gigante, a seção 17.1 mediu que padronizar quase não mexia no peso da área na distância, e que o logaritmo antes da padronização tirava dela o monopólio. Para os anúncios comuns, o efeito da cauda longa é outro: os poucos valores enormes inflam o desvio padrão, e, divididos por ele, os valores comuns ficam espremidos perto da média. O logaritmo encurta a cauda antes da padronização, e por isso as duas etapas ficam juntas, encadeadas num `Pipeline`:

In [ ]:
assimetricas = Pipeline([
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("escala", StandardScaler()),
])

> **🔧 Função**
>
> **`FunctionTransformer(funcao, feature_names_out)`** — transforma cada coluna com uma função fixa, aqui `np.log1p`. O `fit` não aprende nada. `feature_names_out="one-to-one"` avisa que cada coluna de saída corresponde a uma de entrada, com o mesmo nome.
>
> **`StandardScaler()`** — a padronização: no `fit`, guarda a média e o desvio padrão de cada coluna; no `transform`, subtrai a média e divide pelo desvio.
>
> **`Pipeline(etapas)`** — encadeia etapas, cada uma um par `(nome, objeto)`: a saída de uma é a entrada da seguinte. No `fit`, cada etapa aprende a partir da saída da anterior.

**O `andar`.** Pelos indícios da seção 6.4, o nulo de `andar` marca uma casa, mas a seção 17.1 contou anúncios sem andar e com condomínio, que tanto podem ser casas em condomínio fechado quanto apartamentos sem o andar informado. Preencher só com 0, como na seção 6.4, põe esses anúncios num andar que o arquivo não tem:

In [ ]:
print("menor andar no treino:", numero_br(X_treino["andar"].min(), 0))

Com o 0 logo abaixo do 1º andar, o *k*-NN põe a casa entre os vizinhos dos apartamentos do 1º andar, e a regressão linear prende o nível da casa à tendência do andar, prolongada até o 0. Um indicador desfaz os dois vínculos: o nulo vira 0, e uma coluna nova diz se o andar estava vazio. O modelo pode dar aos anúncios sem andar um nível próprio, sem que se precise decidir qual é casa. Dois anúncios com andar e dois sem, como estão no arquivo:

In [ ]:
preenche = SimpleImputer(strategy="constant", fill_value=0, add_indicator=True)
preenche.set_output(transform="pandas").fit(X_treino[["andar"]])
amostra = X_treino[["andar"]].groupby(X_treino["andar"].isna()).head(2)
amostra

E os mesmos quatro depois do preenchimento:

In [ ]:
preenche.transform(amostra).astype(int)

> **🔧 Função**
>
> **`SimpleImputer(strategy, fill_value, add_indicator)`** — preenche nulos. Com `strategy="constant"`, preenche com `fill_value`, aqui 0; outras estratégias usam a média ou a mediana aprendidas no `fit`. Com `add_indicator=True`, acrescenta a coluna `missingindicator_andar`, com 1 onde o valor estava nulo.
>
> **`df.groupby(chave).head(n)`** — as `n` primeiras linhas de cada grupo. Aqui a chave é `isna()`, e vêm dois anúncios com andar e dois sem.

Depois do preenchimento, as duas colunas passam pela padronização, como as demais numéricas:

In [ ]:
andar = Pipeline([
    ("preenche", SimpleImputer(strategy="constant", fill_value=0, add_indicator=True)),
    ("escala", StandardScaler()),
])

**As contagens.** `quartos`, `banheiros` e `vagas` passam só pela padronização, pelo mesmo motivo das outras numéricas.

O `ColumnTransformer` junta as quatro transformações, cada uma com a sua lista de colunas, e cola os resultados lado a lado:

In [ ]:
preparo = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     ["cidade", "aceita_animal", "mobiliado"]),
    ("assimetricas", assimetricas, ["area_m2", "condominio", "iptu"]),
    ("andar", andar, ["andar"]),
    ("contagens", StandardScaler(), ["quartos", "banheiros", "vagas"]),
]).set_output(transform="pandas")

> **🔧 Função**
>
> **`ColumnTransformer(transformacoes)`** — aplica cada transformação só às colunas dela. Cada item é um trio `(nome, transformador, colunas)`, e a saída junta as colunas produzidas, na ordem da lista. Coluna que não está em nenhuma lista fica de fora. O argumento `handle_unknown="ignore"` do `OneHotEncoder` é o assunto da subseção "Categoria nova", adiante.

As colunas de 0 e 1 das categóricas ficam sem padronização. Elas já variam numa faixa de largura 1, comparável à de um desvio padrão, e nenhuma delas tem cauda que estique a escala.

### O que o preparo aprende

> **🔷 Conceito**
>
> **Tudo o que o preparo aprende, aprende no treino.** O `fit` do preparo guarda a lista de categorias de cada coluna de texto e a média e o desvio de cada coluna padronizada, calculados nos anúncios de treino. Depois disso, qualquer tabela passa pelo `transform` com esses números guardados, seja o treino, seja um anúncio novo: nada é recalculado com as linhas que chegam. O teste precisa se comportar como um anúncio que ainda não chegou, e uma média calculada com o teste junto já traria para o modelo informação dele.

Com as peças montadas, o `fit_transform` ajusta o preparo no treino e devolve a tabela convertida. Antes de rodar: quantas colunas ela deve ter? Some as colunas de cada transformação.

In [ ]:
convertido = preparo.fit_transform(X_treino)
print("antes:", X_treino.shape)
print("depois:", convertido.shape)
for nome in preparo.get_feature_names_out():
    print(nome)

> **🔧 Função**
>
> **`preparo.get_feature_names_out()`** — os nomes das colunas de saída, cada um com o nome da transformação que o produziu antes de `__`.

São 17 colunas: cinco de cidade, duas de `aceita_animal` e duas de `mobiliado`, as três assimétricas, `andar` e o indicador, e as três contagens. Os números que o `fit` guardou ficam dentro do preparo, e podem ser lidos:

In [ ]:
aprendido = preparo.named_transformers_
print("cidades vistas no treino:")
for cidade in aprendido["categoricas"].categories_[0]:
    print("  " + cidade)

escala = aprendido["assimetricas"]["escala"]
print("média e desvio depois do log1p:")
for nome, media, desvio in zip(escala.feature_names_in_, escala.mean_, escala.scale_):
    print(f"  {nome}: {numero_br(media)} e {numero_br(desvio)}")

> **🔧 Função**
>
> **`preparo.named_transformers_`** — as transformações já ajustadas, pelo nome dado a cada uma. Num `Pipeline`, `pipeline["nome"]` devolve a etapa com esse nome.
>
> **`codificador.categories_`** — no `OneHotEncoder` ajustado, a lista das categorias de cada coluna, na ordem das colunas de saída.
>
> **`escala.mean_`**, **`escala.scale_`** e **`escala.feature_names_in_`** — no `StandardScaler` ajustado, a média e o desvio padrão de cada coluna e os nomes das colunas, na mesma ordem.

A média de `area_m2` depois do `log1p` é 4,66. Esse número vale para qualquer anúncio que passar pelo preparo, inclusive um que não esteja no treino, como a última subseção confere. O primeiro anúncio de treino, antes e depois:

In [ ]:
X_treino.iloc[0]

In [ ]:
(
    convertido.iloc[0].to_frame("depois").style.set_uuid("primeiro")
    .format(lambda v: numero_br(v, 2))
    .set_table_styles([{"selector": "th, td", "props": "padding: 0.15em 0.3em"}])
)

> **🔧 Função**
>
> **`df.iloc[i]`** — a linha na posição `i` da tabela, contada a partir de 0, como uma `Series` indexada pelos nomes das colunas.
>
> **`serie.to_frame(nome)`** — transforma a `Series` numa tabela de uma coluna, com esse nome.

Este anúncio fica no 1º andar, e o indicador de `andar`, que era 0 para ele, virou −0,55: a padronização também passou pelo indicador. Ele continua com dois valores só, um para os anúncios com andar e outro para os sem, agora em outra escala.

A figura mostra o que o preparo fez com as três colunas assimétricas. Antes de olhar: os maiores valores do treino, como a área de 46.335 m², continuam na tabela depois do preparo?

In [ ]:
# Figura: As três colunas assimétricas nos 8.267 anúncios de treino. À esquerda, como vêm no arquivo, em 40 barras; à direita, depois do log1p e da padronização, em desvios padrão, com barras de 0,25 e o mesmo eixo x nos três painéis da direita. O triângulo cinza marca o maior valor de cada coluna.
colunas_assimetricas = [("area_m2", "área (m²)"),
                        ("condominio", "condomínio (R$)"),
                        ("iptu", "IPTU (R$)")]
em_br = lambda v, pos: numero_br(v, 0)
bordas_depois = np.linspace(-4.5, 8.5, 53)
fig, eixos = plt.subplots(3, 2, figsize=(6.4, 7.2))
for (coluna, rotulo), (ax_antes, ax_depois) in zip(colunas_assimetricas, eixos):
    antes = X_treino[coluna]
    depois = convertido[f"assimetricas__{coluna}"]
    ax_antes.hist(antes, bins=40, color="C0", edgecolor="white", linewidth=0.3)
    ax_antes.set_xlim(0, antes.max() * 1.05)
    ax_antes.xaxis.set_major_locator(MaxNLocator(3))
    ax_antes.xaxis.set_major_formatter(em_br)
    ax_antes.set_xlabel(rotulo)
    ax_antes.set_ylabel("anúncios")
    ax_depois.hist(depois, bins=bordas_depois, color="C0",
                   edgecolor="white", linewidth=0.3)
    ax_depois.set_xlim(-4.5, 8.5)
    ax_depois.set_xticks([-4, -2, 0, 2, 4, 6, 8])
    ax_depois.xaxis.set_major_formatter(em_br)
    ax_depois.set_xlabel(rotulo.split(" (")[0] + " (desvios padrão)")
    for ax, valores in ((ax_antes, antes), (ax_depois, depois)):
        topo = ax.get_ylim()[1]
        ax.scatter(valores.max(), 0.06 * topo, marker="v", s=40, color="0.45",
                   zorder=3, label="_nolegend_")
        ax.yaxis.set_major_formatter(em_br)
eixos[0, 0].set_title("no arquivo")
eixos[0, 1].set_title("depois do preparo")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.linspace(inicio, fim, n)`** — `n` números igualmente espaçados de `inicio` a `fim`. Aqui, as bordas das barras da coluna da direita, as mesmas nos três painéis.
>
> **`ax.set_xticks(lista)`** — fixa as posições das marcas do eixo horizontal; aqui, de −4 a 8, de 2 em 2.
>
> **`ax.xaxis.set_major_locator(MaxNLocator(n))`** — limita o eixo a cerca de `n` marcas, para os rótulos longos em reais não se encostarem. `MaxNLocator` vem de `matplotlib.ticker`.

Os números por trás de cada painel:

In [ ]:
print("mesmas linhas:", len(convertido) == len(X_treino))
for coluna, _ in colunas_assimetricas:
    contagens, _ = np.histogram(X_treino[coluna], bins=40)
    fracao = 100 * contagens[0] / len(X_treino)
    depois = convertido[f"assimetricas__{coluna}"]
    maximo = X_treino[coluna].idxmax()
    print(coluna)
    print("  máximo:", numero_br(X_treino[coluna].max(), 0))
    print(f"  1ª barra: {numero_br(fracao, 1)}%")
    print("  depois:", numero_br(depois.min()), "a", numero_br(depois.max()))
    print("  máximo, depois:", numero_br(depois[maximo]))
    zeros = X_treino[coluna] == 0
    if zeros.any():
        print("  zero, depois:", numero_br(depois[zeros].iloc[0]))

À esquerda, a primeira barra de cada coluna leva quase todos os anúncios: 99,9% na área, 99,2% no condomínio e 99,9% no IPTU. O maior valor de cada coluna (46.335 m² de área, R$ 220.000 de condomínio e R$ 313.700 de IPTU) estica o eixo, e o resto se espreme junto ao zero.

À direita, nenhum extremo saiu, e o maior valor de cada coluna é a ponta direita dela. A área vai de −2,77 a 7,73 e forma um morro, com o imóvel de 46.335 m² isolado no 7,73. O condomínio vai de −1,78 a 2,44, e o IPTU, de −2,02 a 3,74. Nos dois, a ponta esquerda é a dos anúncios com valor zero, que o `log1p` leva ao zero e a padronização leva a −1,78 e a −2,02: é a barra separada à esquerda do morro.

### Categoria nova

O treino tem cinco cidades. O que acontece se chegar um anúncio de Brasília? O codificador do começo da seção, ajustado com o padrão do `OneHotEncoder`, recusa:

In [ ]:
brasilia = pd.DataFrame({"cidade": ["Brasília"]})
try:
    codificador.transform(brasilia)
except ValueError as erro:
    print(textwrap.fill(str(erro)[:120], width=36))

> **🔧 Função**
>
> **`textwrap.fill(texto, width)`** — quebra o texto em linhas de até `width` caracteres, do módulo `textwrap` da biblioteca padrão. Aqui, só para a mensagem caber na página.

A mensagem diz que há uma categoria desconhecida, `Brasília`, na coluna 0. O padrão é `handle_unknown="error"`, e é seguro: um anúncio de uma cidade para a qual o modelo não aprendeu nada não recebe previsão sem que alguém perceba. O preparo usa `handle_unknown="ignore"`, que deixa o anúncio passar com as cinco colunas de cidade em zero:

In [ ]:
ignora = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ignora.set_output(transform="pandas").fit(X_treino[["cidade"]])
ignora.transform(brasilia).T.astype(int)

Com as cinco colunas em zero, o anúncio de Brasília passa pelo preparo e recebe uma previsão, mas o treino não tem um anúncio de lá, e a subseção seguinte mostra o que esse número é na regressão linear. O custo do `"ignore"` é o silêncio: qualquer nome fora da lista das cinco, inclusive um erro de digitação, passa da mesma forma, sem aviso. Qual dos dois comportamentos você escolheria para uma plataforma que publica a sugestão de preço sem revisão humana?

### O Pipeline: preparo e modelo como um objeto só

O preparo e o modelo podem ficar num mesmo `Pipeline`. O `fit` do conjunto ajusta o preparo no treino, converte o treino e ajusta o modelo na tabela convertida; o `predict` converte a tabela que chega com os números guardados e passa o resultado ao modelo. A regressão linear múltipla da seção 8.3 ocupa o passo `modelo`:

In [ ]:
modelo = Pipeline([("preparo", preparo), ("modelo", LinearRegression())])
modelo = modelo.fit(X_treino, y_treino)
print("o preparo é o mesmo objeto:", modelo["preparo"] is preparo)

> **🔧 Função**
>
> **`LinearRegression()`** — a regressão linear por mínimos quadrados; o `fit` estima o intercepto e um coeficiente por coluna.

Um anúncio inventado: uma casa em São Paulo, sem andar, com 120 m², três quartos, dois banheiros, duas vagas, que aceita animal, não mobiliada, sem condomínio e com R$ 150 de IPTU. O anúncio é uma tabela de uma linha com as mesmas colunas de `X_treino`:

In [ ]:
anuncio = pd.DataFrame({
    "cidade": ["São Paulo"], "area_m2": [120], "quartos": [3], "banheiros": [2],
    "vagas": [2], "andar": [np.nan], "aceita_animal": ["sim"],
    "mobiliado": ["não"], "condominio": [0], "iptu": [150],
})
log_previsto = modelo.predict(anuncio)[0]
print("previsão em log:", numero_br(log_previsto, 4))
print("em reais: R$", numero_br(np.exp(log_previsto), 0))

> **🔧 Função**
>
> **`modelo.predict(X)`** — uma previsão para cada linha de `X`, depois de passar `X` pelo preparo. Aqui, o logaritmo do aluguel; como `X` tem uma linha só, o `[0]` tira o único número do array.
>
> **`np.exp(x)`** — a exponencial, o inverso de `np.log`: `np.exp(np.log(x))` devolve `x`. Aqui, leva a previsão de volta a reais.

O modelo prevê o logaritmo do aluguel, 7,9988, e `np.exp` leva o número de volta a reais: R$ 2.977 por mês é a sugestão para a casa inventada.

O `predict` converteu o anúncio com a média e o desvio do treino, e não com números do próprio anúncio: numa tabela de uma linha, a média de cada coluna é o próprio valor do anúncio, e padronizar com ela levaria toda coluna padronizada a zero, qualquer que fosse o anúncio. A conta pode ser refeita com os números que o `fit` guardou, a média 4,66 de `area_m2` depois do `log1p` e o desvio correspondente:

In [ ]:
convertido_anuncio = modelo["preparo"].transform(anuncio).iloc[0]
escala = modelo["preparo"].named_transformers_["assimetricas"]["escala"]
media, desvio = escala.mean_[0], escala.scale_[0]
pelo_preparo = convertido_anuncio["assimetricas__area_m2"]
a_mao = (np.log1p(120) - media) / desvio
print("área, pelo preparo:", numero_br(pelo_preparo, 4))
print("área, à mão:", numero_br(a_mao, 4))
print("iguais:", np.isclose(pelo_preparo, a_mao))
print("andar:", numero_br(convertido_anuncio["andar__andar"]))
print("indicador:", numero_br(convertido_anuncio["andar__missingindicator_andar"]))

> **🔧 Função**
>
> **`np.isclose(a, b)`** — `True` quando `a` e `b` são iguais a menos do arredondamento do ponto flutuante.

O `modelo["preparo"]` é o mesmo objeto `preparo`, como o booleano logo depois do `fit` confirmou. Ele foi reajustado dentro do `fit` do `Pipeline`, com os mesmos anúncios de treino. A casa inventada recebeu o andar 0 e o indicador de andar vazio, os dois já na escala do treino: −0,82 e 1,82. O indicador, que era −0,55 no anúncio do 1º andar, é positivo nela.

E a mesma casa em Brasília? Ela passa pelo preparo com as cinco colunas de cidade em zero, e o modelo devolve um número. Para comparar, a média das previsões para a casa em cada uma das cinco cidades do treino:

In [ ]:
cidades_vistas = modelo["preparo"].named_transformers_["categoricas"].categories_[0]
em_cada = [modelo.predict(anuncio.assign(cidade=c))[0] for c in cidades_vistas]
em_brasilia = modelo.predict(anuncio.assign(cidade="Brasília"))[0]
print("Brasília:", numero_br(em_brasilia, 4))
print("média das cinco:", numero_br(np.mean(em_cada), 4))

> **🔧 Função**
>
> **`np.mean(valores)`** — a média de uma lista de números.

A casa em Brasília recebe 7,7496, a média das cinco cidades. O número não vem de nada que o treino ensinou sobre Brasília, e sim de como o ajuste repartiu o efeito da cidade entre o intercepto e as cinco colunas. Outra codificação mostra isso: com `drop="first"`, cada categórica perde a coluna da primeira categoria, e Belo Horizonte passa a ser a cidade sem coluna. As previsões para as cidades conhecidas não mudam:

In [ ]:
outra = clone(modelo).set_params(preparo__categoricas__drop="first")
outra = outra.fit(X_treino, y_treino)
iguais = np.allclose(outra.predict(X_treino), modelo.predict(X_treino))
print("mesmas previsões no treino:", iguais)
print("São Paulo:", numero_br(outra.predict(anuncio)[0], 4))
belo_horizonte = anuncio.assign(cidade="Belo Horizonte")
em_bh = outra.predict(belo_horizonte)[0]
print("Belo Horizonte:", numero_br(em_bh, 4))
with warnings.catch_warnings():
    # o scikit-learn avisa que "Brasília" é categoria desconhecida,
    # codificada com todas as colunas em zero: é o que se quer ver aqui
    warnings.simplefilter("ignore", UserWarning)
    em_brasilia_outra = outra.predict(anuncio.assign(cidade="Brasília"))[0]
print("Brasília:", numero_br(em_brasilia_outra, 4))
print("igual a Belo Horizonte:", em_brasilia_outra == em_bh)

> **🔧 Função**
>
> **`clone(modelo)`** — uma cópia do modelo com os mesmos parâmetros, ainda sem ajuste. Vem de `sklearn.base`.
>
> **`modelo.set_params(etapa__parametro=valor)`** — troca um parâmetro; o `__` desce pelas etapas pelo nome. Aqui, o `drop` do `OneHotEncoder` da transformação `categoricas`, dentro do `preparo`.
>
> **`np.allclose(a, b)`** — `True` quando todos os pares de valores são iguais a menos do arredondamento.
>
> **`warnings.catch_warnings()`** — num bloco `with`, isola os filtros de aviso: o que se muda dentro dele vale só ali. **`warnings.simplefilter("ignore", UserWarning)`** cala os avisos desse tipo; os dois vêm do módulo `warnings` da biblioteca padrão.

A casa em São Paulo continua em 7,9988. Nessa codificação, um anúncio de Brasília tem as quatro colunas de cidade em zero, exatamente como um de Belo Horizonte, e recebe os mesmos 7,6944 de Belo Horizonte, e não os 7,7496 de antes.

Os dois ajustes são igualmente bons nas cinco cidades conhecidas, com as mesmas previsões para todo anúncio de treino; o que os separa é só a codificação, e nada no dado escolhe entre elas. Por isso a previsão para uma cidade que o treino não viu não é confiável, qualquer que seja o número. É esse o custo do `"ignore"` visto de dentro do modelo: ele só é aceitável se alguém revisar as categorias que chegam, para que uma cidade nova seja notada antes de a previsão ser publicada.

Com o preparo dentro do `Pipeline`, não há como ajustar a escala numa tabela e o modelo em outra por descuido: um `fit` ajusta os dois, com as mesmas linhas. A validação cruzada da seção 10.6 reajusta essa estrutura inteira em cada grupo, e o `StandardScaler` aprende só das linhas de treino do grupo: é assim que o vazamento de pré-processamento fica de fora.

## Comparando Modelos por Validação Cruzada

Três candidatos disputam a sugestão de preço: prever para todo anúncio o mesmo número, a média; a regressão linear; e o *k*-NN com dez vizinhos. Qual deles erra menos num anúncio que ainda não chegou? O teste guardado responderia, mas ele só pode ser aberto uma vez, para o modelo já escolhido. A escolha precisa sair do treino.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import (KFold, cross_val_predict, cross_val_score,
                                     train_test_split)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

O ponto de partida é a tabela sem linhas repetidas, a divisão estratificada por cidade, os dez preditores e o logaritmo do aluguel como resposta. O `preparo` é o da seção 17.3: as colunas categóricas viram colunas de 0 e 1; área, condomínio e IPTU passam por `np.log1p` e são padronizados; o andar ausente vira 0, com uma coluna que marca onde ele faltava; e as contagens são padronizadas.

In [ ]:
alugueis = pd.read_csv("dados/alugueis.csv", na_values=["-"]).drop_duplicates()
preditores = ["cidade", "area_m2", "quartos", "banheiros", "vagas", "andar",
              "aceita_animal", "mobiliado", "condominio", "iptu"]
treino, teste = train_test_split(alugueis, test_size=0.2, random_state=17,
                                 stratify=alugueis["cidade"])
X_treino, y_treino = treino[preditores], np.log(treino["aluguel"])

preparo = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     ["cidade", "aceita_animal", "mobiliado"]),
    ("assimetricas", Pipeline([
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("escala", StandardScaler())]), ["area_m2", "condominio", "iptu"]),
    ("andar", Pipeline([
        ("preenche", SimpleImputer(strategy="constant", fill_value=0,
                                   add_indicator=True)),
        ("escala", StandardScaler())]), ["andar"]),
    ("contagens", StandardScaler(), ["quartos", "banheiros", "vagas"]),
]).set_output(transform="pandas")
print("X_treino:", X_treino.shape)

### Os grupos

A validação cruzada divide o treino em grupos, e cada grupo, na sua vez, fica de fora enquanto os outros ajustam o modelo, como na seção 10.3. Os grupos são sorteados: `shuffle=True` embaralha os anúncios antes de cortar, com a semente 17. Os mesmos dez grupos servem para os três modelos comparados a seguir.

In [ ]:
grupos = KFold(10, shuffle=True, random_state=17)
tamanhos = [len(validacao) for _, validacao in grupos.split(X_treino)]
print("grupos:", len(tamanhos))
print("menor grupo:", min(tamanhos))
print("maior grupo:", max(tamanhos))

> **🔧 Função**
>
> **`KFold(n_splits, shuffle, random_state)`** — o esquema de divisão da validação cruzada em `n_splits` grupos. Com `shuffle=True`, as linhas são embaralhadas antes do corte, com a semente `random_state`; sem ele, o primeiro grupo seria as primeiras linhas da tabela.
>
> **`grupos.split(X)`** — produz, para cada grupo, o par (posições de treino, posições de validação). Aqui só se mede o tamanho de cada grupo de validação.

Os 8.267 anúncios de treino não se dividem por 10 sem resto, e cada grupo fica com 826 ou 827 anúncios.

### Em que escala medir

A resposta poderia ficar em reais, sem o logaritmo. A validação cruzada mostra o que acontece nesse caso. `cross_val_predict` devolve, para cada anúncio de treino, a previsão do modelo ajustado nos outros nove grupos, isto é, sem o grupo em que o anúncio estava. A regressão linear, com a resposta em reais:

In [ ]:
linear_em_reais = Pipeline([("preparo", preparo), ("modelo", LinearRegression())])
previsto = cross_val_predict(linear_em_reais, X_treino, treino["aluguel"], cv=grupos)
print("previsões:", numero_br(len(previsto), 0))
print("negativas:", numero_br((previsto < 0).sum(), 0))
print("menor: R$", numero_br(previsto.min()))

> **🔧 Função**
>
> **`cross_val_predict(estimador, X, y, cv)`** — ajusta o estimador uma vez por grupo de `cv` e devolve uma previsão para cada linha de `X`, feita pelo ajuste em que a linha estava na validação. Com um `Pipeline`, cada ajuste reajusta o preparo junto, só com os outros grupos.

São 159 previsões negativas em 8.267, o menor deles de R$ −1.772,47. De que anúncios eles vêm?

In [ ]:
colunas = ["area_m2", "condominio"]
negativas = X_treino.loc[previsto < 0, colunas].median()
todas = X_treino[colunas].median()
print("mediana: negativas / treino")
print("  área (m²):", numero_br(negativas["area_m2"], 0), "/",
      numero_br(todas["area_m2"], 0))
print("  condomínio (R$):", numero_br(negativas["condominio"], 0), "/",
      numero_br(todas["condominio"], 0))

As previsões negativas vêm de anúncios pequenos: área mediana de 26 m² contra 95 no treino inteiro, condomínio mediano de R$ 140 contra R$ 568. Uma reta em reais não tem piso, e nada impede que, num anúncio pequeno o bastante, a soma dos termos fique abaixo de zero. Por que a regressão linear sobre o logaritmo do aluguel não pode sugerir um preço negativo, qualquer que seja o anúncio?

Na escala log, a previsão é $\log \hat y$, e o preço sugerido é $\hat y = e^{\log \hat y}$, positivo para qualquer valor do expoente. O erro também muda de natureza. A raiz do erro quadrático médio (RMSE, em inglês *root mean squared error*) na escala log é

$$
\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(\log y_i - \log \hat y_i\right)^2},
$$

em que $\log$ é o logaritmo natural, o `np.log`. Cada diferença $\log y_i - \log \hat y_i$ é $\log(y_i / \hat y_i)$, o logaritmo da razão entre o aluguel e a previsão. Considere prever R$ 1.000 para um aluguel de R$ 1.500 e R$ 10.000 para um de R$ 15.000:

In [ ]:
print("log(1.500 / 1.000):", numero_br(np.log(1500 / 1000), 4))
print("log(15.000 / 10.000):", numero_br(np.log(15000 / 10000), 4))

Em reais, um erro é de R$ 500 e o outro de R$ 5.000; na escala log, os dois valem o mesmo, porque a previsão ficou abaixo do aluguel pelo mesmo fator, 1,5. Errar pelo mesmo fator conta o mesmo num aluguel barato e num caro, como a seção 17.2 mediu. É na escala log que os três candidatos são comparados: é nela que eles são ajustados, e nela o erro é relativo.

### Três candidatos

Os três candidatos entram no mesmo `Pipeline`, depois do mesmo preparo, e mudam só no passo `modelo`:

In [ ]:
candidatos = {
    "média": DummyRegressor(),
    "linear": LinearRegression(),
    "k-NN": KNeighborsRegressor(n_neighbors=10),
}

> **🔧 Função**
>
> **`DummyRegressor()`** — um "modelo" que ignora os preditores: no `fit`, guarda a média da resposta; no `predict`, devolve essa média para todo anúncio. Serve de referência: um modelo que não erra menos que ele não aprendeu nada dos preditores.

A média não usa o preparo, mas passar por ele não muda nada nela, e assim os três são tratados igualmente. Antes de rodar: quanto a média deve errar? Ela prevê o mesmo número para todo anúncio, e o RMSE de uma previsão constante igual à média é aproximadamente o desvio padrão da resposta.

In [ ]:
print("desvio padrão de y_treino:", numero_br(y_treino.std(ddof=0), 4))

> **🔧 Função**
>
> **`serie.std(ddof=0)`** — o desvio padrão dividindo por $n$, e não por $n - 1$, que é o padrão do `pandas`. Com `ddof=0`, é exatamente a raiz da média dos quadrados das distâncias à média.

O `cross_val_score` ajusta cada `Pipeline` nos dez pares de grupos e devolve um erro por grupo:

In [ ]:
rmse = pd.DataFrame({
    nome: -cross_val_score(Pipeline([("preparo", preparo), ("modelo", modelo)]),
                           X_treino, y_treino, cv=grupos,
                           scoring="neg_root_mean_squared_error")
    for nome, modelo in candidatos.items()
}, index=range(1, 11))
rmse.style.format(lambda v: numero_br(v, 4))

> **🔧 Função**
>
> **`cross_val_score(estimador, X, y, cv, scoring)`** — para cada grupo de `cv`, ajusta o estimador nos outros e mede o erro no grupo; devolve um número por grupo. `scoring="neg_root_mean_squared_error"` pede o RMSE com o sinal trocado: o `scikit-learn` trata toda pontuação como "maior é melhor", e o `-` na frente desfaz a troca.

Cada linha é um grupo de validação, e as três colunas foram medidas nos mesmos anúncios. As médias dos dez grupos, o desvio padrão entre eles e o fator correspondente, $e^{\text{média}}$:

In [ ]:
for nome in rmse:
    media = rmse[nome].mean()
    print(nome)
    print("  média:", numero_br(media, 4))
    print("  desvio entre grupos:", numero_br(rmse[nome].std(), 4))
    print("  fator e^média:", numero_br(np.exp(media), 2))

A média erra 0,7902, e o desvio padrão de `y_treino` também é 0,7902: os dois coincidem até a quarta casa, como esperado. A regressão linear erra 0,4458 e o *k*-NN, 0,4217, os dois bem abaixo da média, e o *k*-NN um pouco abaixo da linear.

Para ler o erro em preços, desfaz-se o logaritmo natural com a exponencial: o fator do *k*-NN, $e^{0{,}4217}$, é 1,52, e o da média é 2,20. Um fator de 1,52 é uma previsão 52% acima do aluguel ou, dividindo por 1,52, 34% abaixo dele (1 − 1/1,52).

Esse fator é um erro típico, e não o erro de um anúncio comum: o RMSE eleva as diferenças ao quadrado antes de tirar a média, e os anúncios mais mal previstos o puxam para cima.

### Uma comparação pareada

A diferença entre a linear e o *k*-NN, 0,0241 (0,4458 − 0,4217), é só cerca do dobro do desvio padrão entre os grupos de cada um, 0,0113 na linear e 0,0127 no *k*-NN, e os erros dos dois se sobrepõem:

In [ ]:
pior_knn, melhor_linear = rmse["k-NN"].max(), rmse["linear"].min()
print("pior grupo do k-NN:", numero_br(pior_knn, 4))
print("melhor grupo da linear:", numero_br(melhor_linear, 4))
print("se sobrepõem:", bool(pior_knn > melhor_linear))

Olhando as duas colunas como dois montes de dez números, o erro do *k*-NN em alguns grupos é maior que o da linear em outros. Mas os dois modelos foram medidos nos **mesmos** grupos, e um grupo com anúncios difíceis é difícil para os dois. A figura abaixo liga, por uma linha cinza, os três erros de cada grupo. No código, `ax.plot` recebe a tabela `rmse[nomes].T`, que tem uma coluna por grupo, e desenha uma linha para cada coluna; `ax.set_xticks(posicoes, nomes)` põe o nome de cada modelo sob a sua posição no eixo horizontal:

In [ ]:
# Figura: RMSE na escala log dos três candidatos nos dez grupos da validação cruzada; a linha cinza liga os erros do mesmo grupo, e o traço horizontal marca a média dos dez. Em cima, os três modelos, com o eixo desde zero; embaixo, só a linear e o k-NN, com o eixo ampliado.
cores = {"média": "0.45", "linear": "C0", "k-NN": "C1"}
em_br = lambda v, pos: numero_br(v, 2)
fig, (ax_todos, ax_zoom) = plt.subplots(2, 1, figsize=(6.4, 6.4))
for ax, nomes in ((ax_todos, list(cores)), (ax_zoom, ["linear", "k-NN"])):
    posicoes = list(range(len(nomes)))
    ax.plot(posicoes, rmse[nomes].T, color="0.45", linewidth=0.8, alpha=0.5)
    for x, nome in zip(posicoes, nomes):
        ax.scatter([x] * len(rmse), rmse[nome], s=36, color=cores[nome], zorder=3,
                   label="_nolegend_")
        ax.hlines(rmse[nome].mean(), x - 0.25, x + 0.25, color=cores[nome],
                  linewidth=2.5, zorder=2)
    ax.set_xticks(posicoes, nomes)
    ax.set_xlim(-0.5, len(nomes) - 0.5)
    ax.set_ylabel("RMSE (escala log)")
    ax.yaxis.set_major_formatter(em_br)
ax_todos.set_ylim(0, 0.9)
ax_todos.set_title("os três candidatos")
ax_zoom.set_title("linear e k-NN, eixo ampliado")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.hlines(y, inicio, fim, color, linewidth)`** — um traço horizontal na altura `y`, de `inicio` a `fim` no eixo horizontal.

No painel de cima, a queda da média para a linear é o ganho que os preditores trazem, e ela acontece em todos os grupos. No de baixo, cada linha desce da linear para o *k*-NN. A conta, grupo a grupo:

In [ ]:
ganho = rmse["linear"] - rmse["k-NN"]
print("linear abaixo da média nos 10:", bool((rmse["linear"] < rmse["média"]).all()))
print("k-NN abaixo da linear nos 10:", bool((ganho > 0).all()))
print("ganho do k-NN, menor:", numero_br(ganho.min(), 4))
print("ganho do k-NN, maior:", numero_br(ganho.max(), 4))
print("desvio do ganho:", numero_br(ganho.std(), 4))
print("correlação dos erros:", numero_br(rmse["linear"].corr(rmse["k-NN"]), 2))

> **🔧 Função**
>
> **`serie.corr(outra)`** — a correlação entre duas colunas, linha a linha; aqui, entre os erros da linear e do *k*-NN nos mesmos dez grupos.

O *k*-NN erra menos que a linear nos dez grupos. O ganho vai de 0,0043 a 0,0365, e o desvio padrão dele entre os grupos, 0,0090, é menor que o de cada modelo sozinho, 0,0113 na linear e 0,0127 no *k*-NN. Os erros dos dois sobem e descem juntos de um grupo para outro, com correlação de 0,72: parte do que varia é a dificuldade do grupo, que atinge os dois modelos e, na diferença, em boa parte se cancela. A evidência a favor do *k*-NN é uma diferença com o mesmo sinal nos dez grupos, e não só a das médias.

Ela tem limites. Os dez ajustes usam quase os mesmos anúncios de treino, então os dez ganhos não são dez medições independentes. E o ganho é pequeno perto do que a linear ganha da média: 0,0241 (0,4458 − 0,4217) contra 0,3444 (0,7902 − 0,4458), e por isso a linha que liga os dois modelos é quase plana no painel de cima.

E se o *k*-NN ganhasse em só quatro dos dez grupos? Aí a vantagem deixaria de convencer: a diferença entre os dois seria do tamanho da variação de um grupo de anúncios para outro, e outros anúncios poderiam inverter a ordem.

Os dez grupos são um sorteio, e outra semente daria outros. Com um ganho que nunca troca de sinal nos dez grupos da semente 17, você esperaria que ele trocasse com outra semente? A mesma comparação, repetida com as sementes de 0 a 4:

In [ ]:
vitorias, ganhos_medios = [], []
for semente in range(5):
    outros_grupos = KFold(10, shuffle=True, random_state=semente)
    erro = {nome: -cross_val_score(
                Pipeline([("preparo", preparo), ("modelo", candidatos[nome])]),
                X_treino, y_treino, cv=outros_grupos,
                scoring="neg_root_mean_squared_error")
            for nome in ["linear", "k-NN"]}
    ganho_semente = erro["linear"] - erro["k-NN"]
    vitorias.append(int((ganho_semente > 0).sum()))
    ganhos_medios.append(ganho_semente.mean())
    print(f"semente {semente}: k-NN vence em", vitorias[-1], "de 10")
    print("  ganho médio:", numero_br(ganhos_medios[-1], 4))
print("vence nos 10 em todas:", all(v == 10 for v in vitorias))
print("ganho médio, menor:", numero_br(min(ganhos_medios), 4))
print("ganho médio, maior:", numero_br(max(ganhos_medios), 4))

Nas cinco sementes, o *k*-NN erra menos que a linear nos dez grupos, e o ganho médio fica entre 0,0243 e 0,0258, perto dos 0,0241 da semente 17. O *k*-NN fica à frente e segue como candidato, e o teste continua fechado.

Nada na comparação depende de que modelos são esses. O passo `modelo` do `Pipeline` aceita qualquer regressor do `scikit-learn`, inclusive as árvores de decisão e as florestas dos capítulos 12 e 13, e o mesmo `cross_val_score`, com os mesmos grupos, os colocaria na tabela. Os dez vizinhos do *k*-NN foram fixados de antemão, e esse número também pode ser escolhido por validação cruzada.

## Ajuste de Hiperparâmetros com GridSearchCV

Quantos vizinhos o *k*-NN deve consultar? E área, condomínio e IPTU devem passar pelo logaritmo antes da padronização? Nada no dado pediu os dez vizinhos da seção 17.4, nem o logaritmo, e as duas escolhas mudam o erro do modelo. Como escolher as duas, e com que dado, sem abrir o teste?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

O ponto de partida é a tabela sem linhas repetidas, a divisão estratificada por cidade, os dez preditores e o logaritmo do aluguel como resposta. O `preparo` é o da seção 17.3: as colunas categóricas viram colunas de 0 e 1; área, condomínio e IPTU passam por `np.log1p` e são padronizados; o andar ausente vira 0, com uma coluna que marca onde ele faltava; e as contagens são padronizadas.

In [ ]:
alugueis = pd.read_csv("dados/alugueis.csv", na_values=["-"]).drop_duplicates()
preditores = ["cidade", "area_m2", "quartos", "banheiros", "vagas", "andar",
              "aceita_animal", "mobiliado", "condominio", "iptu"]
treino, teste = train_test_split(alugueis, test_size=0.2, random_state=17,
                                 stratify=alugueis["cidade"])
X_treino, y_treino = treino[preditores], np.log(treino["aluguel"])

preparo = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     ["cidade", "aceita_animal", "mobiliado"]),
    ("assimetricas", Pipeline([
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("escala", StandardScaler())]), ["area_m2", "condominio", "iptu"]),
    ("andar", Pipeline([
        ("preenche", SimpleImputer(strategy="constant", fill_value=0,
                                   add_indicator=True)),
        ("escala", StandardScaler())]), ["andar"]),
    ("contagens", StandardScaler(), ["quartos", "banheiros", "vagas"]),
]).set_output(transform="pandas")
print("X_treino:", X_treino.shape)

### Escolhas que o `fit` não faz

O `fit` do *k*-NN guarda os anúncios de treino, e o do `StandardScaler` aprende uma média e um desvio. Nenhum dos dois decide quantos vizinhos consultar nem se a área passa pelo logaritmo antes da escala: essas escolhas são fixadas antes do `fit` e ficam de fora dele.

> **🔷 Conceito**
>
> O que o `fit` aprende do dado são os **parâmetros** do modelo: a média e o desvio de cada coluna, no `StandardScaler`; os anúncios guardados, no *k*-NN; os coeficientes, na regressão linear. Uma escolha fixada antes do `fit`, que ele não altera, é um **hiperparâmetro**: o número de vizinhos do *k*-NN, ou a decisão de passar a área pelo logaritmo antes da escala.

Por que não escolher o número de vizinhos pelo erro no próprio treino? Pense no que acontece com *k* = 1 antes de ver a saída:

In [ ]:
um_vizinho = Pipeline([("preparo", preparo),
                       ("modelo", KNeighborsRegressor(n_neighbors=1))])
um_vizinho.fit(X_treino, y_treino)
residuo = y_treino - um_vizinho.predict(X_treino)
print("RMSE no treino, k = 1:", numero_br(np.sqrt(np.mean(residuo**2)), 4))

> **🔧 Função**
>
> **`np.sqrt(x)`** — a raiz quadrada, elemento a elemento. Com `np.mean(residuo**2)` dentro, é a fórmula do RMSE da seção 17.4.

No treino, o vizinho mais próximo de cada anúncio é ele mesmo, ou outro com os mesmos preditores, e o erro fica perto de zero, em 0,0290. Escolhido pelo erro de treino, o *k* tenderia a cair no 1, e nenhum anúncio novo tem uma cópia de si mesmo à espera no treino. O critério é o erro de validação cruzada, nos mesmos dez grupos da seção 17.4.

### Uma grade sobre o `Pipeline` inteiro

O procedimento é o da seção 11.5, lá para o $\lambda$ da ridge, e tem quatro passos: montar uma grade de valores; medir o erro de validação cruzada de cada um; ficar com o de menor erro; reajustar o modelo com esse valor, agora com todo o treino. O `GridSearchCV` faz os quatro.

A novidade aqui é que a grade não mexe só no modelo. Ela percorre dois hiperparâmetros, um de cada passo do `Pipeline`, e cada um é chamado pelo caminho até ele, com `__` entre os nomes, como no `set_params` da seção 17.3. Os nomes disponíveis saem de `get_params`:

In [ ]:
knn = Pipeline([("preparo", preparo), ("modelo", KNeighborsRegressor())])
for nome in knn.get_params():
    if nome.endswith(("__log", "__n_neighbors")):
        print(nome)

> **🔧 Função**
>
> **`knn.get_params()`** — um dicionário com todas as escolhas fixadas antes do `fit`, isto é, os hiperparâmetros; os parâmetros que o `fit` aprende não estão nele, apesar do nome do método; num `Pipeline`, inclusive os das etapas internas, com o caminho até cada um separado por `__`. Aqui só se imprimem os nomes que terminam em `__log` ou `__n_neighbors`.

`preparo__assimetricas__log` é a etapa `log` do `Pipeline` `assimetricas`, dentro do `preparo`. Ela aceita um transformador no lugar do atual ou a palavra `"passthrough"`, que tira a etapa do caminho: área, condomínio e IPTU vão direto para a padronização, em metros quadrados e em reais. A grade tem doze valores de *k*, de 5 a 60 de 5 em 5, e as duas versões do preparo:

In [ ]:
grupos = KFold(10, shuffle=True, random_state=17)
grade = {
    "modelo__n_neighbors": list(range(5, 61, 5)),
    "preparo__assimetricas__log": [
        FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
        "passthrough",
    ],
}
candidatos = len(grade["modelo__n_neighbors"]) * len(
    grade["preparo__assimetricas__log"])
print("candidatos:", candidatos)
print("ajustes:", candidatos * grupos.get_n_splits() + 1)

> **🔧 Função**
>
> **`"passthrough"`** — no lugar de uma etapa de um `Pipeline`, pula a etapa: o que entra nela sai sem mudança.
>
> **`grupos.get_n_splits()`** — o número de grupos do esquema de validação cruzada, 10 aqui.

São 24 combinações, cada uma ajustada nos dez grupos, mais o reajuste final do melhor candidato com todo o treino. As 24 são o produto 12 × 2 dos tamanhos das duas listas, e é esse o limite da busca em grade: cada hiperparâmetro novo multiplica o número de candidatos pelo tamanho da sua lista, e o custo junto. Por isso a grade fica restrita ao que tem motivo para variar.

Antes de rodar: o logaritmo foi motivado na seção 17.1 por uma conta sobre dois imóveis, um de área gigante e outro mediano. Ele deve baixar o erro de validação cruzada do *k*-NN em todo *k*? E você esperaria o melhor *k* abaixo ou acima de dez?

In [ ]:
busca = GridSearchCV(knn, grade, cv=grupos, scoring="neg_root_mean_squared_error")
busca.fit(X_treino, y_treino)
resultados = pd.DataFrame({
    "k": busca.cv_results_["param_modelo__n_neighbors"].astype(int),
    "preparo": ["sem log" if isinstance(v, str) else "com log"
                for v in busca.cv_results_["param_preparo__assimetricas__log"]],
    "rmse": -busca.cv_results_["mean_test_score"],
})
erro = resultados.pivot(index="k", columns="preparo", values="rmse")
(erro.rename_axis(index=None, columns="k")
     .style.format(lambda v: numero_br(v, 4))
     .highlight_min(subset=["com log"], props="font-weight: bold"))

> **🔧 Função**
>
> **`GridSearchCV(estimador, grade, cv, scoring)`** — mede o erro de validação cruzada do `estimador` em cada combinação da `grade`, com os grupos de `cv`, e no `fit` reajusta o melhor candidato com todas as linhas. A grade é um dicionário `{"passo__parametro": valores}`, e as combinações são todas as possíveis.
>
> **`busca.cv_results_`** — um dicionário com uma entrada por medida e um valor por candidato: em `"param_modelo__n_neighbors"`, o *k* de cada um; em `"mean_test_score"`, a média nos dez grupos, com o sinal trocado, como no `cross_val_score`.
>
> **`df.pivot(index, columns, values)`** — rearruma uma tabela longa: uma linha por valor de `index`, uma coluna por valor de `columns`, preenchidas com `values`.
>
> **`df.rename_axis(index, columns)`** — troca o nome do eixo das linhas e o das colunas; aqui, o `k` passa para o canto da tabela, e o cabeçalho fica numa linha só.
>
> **`.highlight_min(subset, props)`** — aplica o estilo `props`, aqui o negrito, à menor célula das colunas de `subset`.

Os candidatos com `"passthrough"` são rotulados pelo tipo, `isinstance(v, str)`, porque o outro valor da grade é um objeto, o `FunctionTransformer`, e não um texto. O `.astype(int)` converte a coluna de *k* em números inteiros, o tipo com que ela é lida nas contas adiante. O negrito marca o menor erro com o logaritmo. O chunk abaixo lê o melhor candidato e o compara com os dez vizinhos da seção 17.4:

In [ ]:
melhor_k = busca.best_params_["modelo__n_neighbors"]
com_log = not isinstance(busca.best_params_["preparo__assimetricas__log"], str)
print("k escolhido:", melhor_k)
print("com log:", com_log)
print("RMSE de validação cruzada:", numero_br(-busca.best_score_, 4))
print("k = 10, com log:", numero_br(erro.loc[10, "com log"], 4))
perto = erro.index[erro["com log"] <= 1.005 * erro["com log"].min()]
print("a 0,5% do mínimo: k de", perto.min(), "a", perto.max())
print("mínimo fora da borda:", bool(5 < melhor_k < 60))

> **🔧 Função**
>
> **`busca.best_params_`** — a combinação da grade com o menor erro, como dicionário.
>
> **`busca.best_score_`** — a pontuação média dessa combinação nos dez grupos, com o sinal trocado.

O escolhido é *k* = 25, acima de dez, com o logaritmo, e o erro de validação cruzada dele é 0,4184. Com os dez vizinhos fixados de antemão, o erro era 0,4217: a busca ganhou 0,0033 (0,4217 − 0,4184). Os valores de *k* de 15 a 40 ficam a menos de 0,5% do mínimo, e o mínimo não está numa ponta da grade, então estendê-la não deve mudar a escolha.

A figura abaixo mostra as 24 medidas. A cor separa as duas versões do preparo:

In [ ]:
# Figura: Erro de validação cruzada (RMSE na escala log) do k-NN para cada número de vizinhos k, com o logaritmo em área, condomínio e IPTU (azul) e sem ele (laranja). O x marca o mínimo, em k = 25 com log; o círculo vazio marca k = 10 com log, o valor fixado de antemão.
cores = {"com log": "C0", "sem log": "C1"}
em_br = lambda v, pos: numero_br(v, 3)
fig, ax = plt.subplots(figsize=(4.8, 3.6))
for preparo_nome, cor in cores.items():
    ax.plot(erro.index, erro[preparo_nome], color=cor, marker="o", markersize=4,
            linewidth=2, label=preparo_nome)
ax.scatter(melhor_k, -busca.best_score_, marker="x", s=90, color="C0",
           linewidths=2.5, zorder=4, label="_nolegend_")
ax.scatter(10, erro.loc[10, "com log"], s=110, facecolors="none",
           edgecolors="0.45", linewidths=2, zorder=4, label="_nolegend_")
ax.set_xticks(range(10, 61, 10))
ax.set_xlim(2, 63)
ax.set_xlabel("k (número de vizinhos)")
ax.set_ylabel("RMSE de validação cruzada")
ax.yaxis.set_major_formatter(em_br)
ax.legend(loc="center right")
plt.tight_layout()
plt.show()

No código, `marker="o"` põe um ponto em cada valor medido da linha, e `facecolors="none"` deixa o círculo vazio, só com o contorno na cor de `edgecolors`.

Duas leituras, com pesos diferentes. O logaritmo ajuda em todo *k* da grade, por uma margem que a conta abaixo mede. O *k*, por sua vez, cai num vale largo: de 15 a 40 a curva azul quase não se mexe.

In [ ]:
ganho_log = erro["sem log"] - erro["com log"]
print("log ganha em todo k:", bool((ganho_log > 0).all()))
print("ganho, menor:", numero_br(ganho_log.min(), 3))
print("ganho, maior:", numero_br(ganho_log.max(), 3))

As duas perguntas de antes da busca têm resposta. Sim, o logaritmo baixa o erro de validação cruzada em todo *k*, com um ganho de 0,022 a 0,028; e o melhor *k*, 25, ficou acima de dez. Mesmo o menor ganho do logaritmo é mais de seis vezes os 0,0033 que a troca de *k* = 10 por *k* = 25 rendeu (0,022 ÷ 0,0033 ≈ 6,7).

Quanto a área pesa na distância entre dois anúncios comuns? A distância do *k*-NN é a raiz da soma dos quadrados das diferenças, coluna a coluna, entre os dois anúncios; o peso de uma coluna é a fatia dela nessa soma. Na seção 17.1, entre o imóvel gigante e um mediano, o logaritmo tirou da área o monopólio da distância. Para os anúncios comuns, o problema sem ele é o avesso desse: os poucos imóveis enormes inflam o desvio padrão da área, e, dividida por esse desvio, a área dos anúncios comuns fica espremida perto da média e quase não pesa na distância entre eles. A conta abaixo mede a fatia mediana da área, com e sem o logaritmo, e também até quantos desvios da média ficam 95% dos anúncios. Os pares são cada linha do treino com a anterior; como a ordem das linhas é a do sorteio da divisão, eles são só uma amostra de pares quaisquer:

In [ ]:
area = X_treino["area_m2"]
print("área, desvio:", numero_br(area.std(), 0), "m²")
print("área, mediana:", numero_br(area.median(), 0), "m²")
sem_log = clone(preparo).set_params(assimetricas__log="passthrough")
for nome, versao in [("com log", clone(preparo)), ("sem log", sem_log)]:
    Z = versao.fit_transform(X_treino)
    quadrados = Z.diff().iloc[1:] ** 2
    fatia = quadrados["assimetricas__area_m2"] / quadrados.sum(axis=1)
    faixa = Z["assimetricas__area_m2"].abs().quantile(0.95)
    print(f"{nome}, 95% até:", numero_br(faixa, 2), "desvio")
    print(f"{nome}, fatia da área:", numero_br(100 * fatia.median(), 1) + "%")

> **🔧 Função**
>
> **`df.diff()`** — a diferença de cada linha para a anterior, coluna a coluna; a primeira linha, sem anterior, fica nula, e o `.iloc[1:]` a descarta.

O desvio padrão da área é de 606 m², contra uma mediana de 95 m². Sem o logaritmo, 95% dos anúncios ficam a menos de 0,41 desvio da média; com ele, a menos de 1,84. Nos pares de anúncios, a fatia mediana da área na soma dos quadrados é de 0,2% sem o logaritmo e de 7,3% com ele. Sem o logaritmo, a área quase não conta na distância entre anúncios comuns, e o *k*-NN escolhe os vizinhos pelas outras colunas; o logaritmo devolve a área à escolha. As duas contas não se contradizem: a da seção 17.1 media um par com o imóvel gigante, e esta mede a mediana de pares comuns; o logaritmo reduz a fatia da área no primeiro caso e a devolve no segundo.

### O vale é raso

Todos os candidatos foram medidos nos mesmos dez grupos, então a comparação pode ser feita grupo a grupo, como na seção 17.4. O `cv_results_` guarda o erro de cada candidato em cada grupo:

In [ ]:
por_grupo = pd.DataFrame(
    [-busca.cv_results_[f"split{g}_test_score"] for g in range(10)],
    columns=list(zip(resultados["k"], resultados["preparo"])),
)
k25_vence_k10 = por_grupo[(melhor_k, "com log")] < por_grupo[(10, "com log")]
log_vence = por_grupo[(melhor_k, "com log")] < por_grupo[(melhor_k, "sem log")]
print(f"erro k = {melhor_k} < erro k = 10:", int(k25_vence_k10.sum()), "de 10")
print("erro com log < sem log:", int(log_vence.sum()), "de 10")

> **🔧 Função**
>
> **`busca.cv_results_[f"split{g}_test_score"]`** — a pontuação de cada candidato no grupo `g`, de 0 a 9. Empilhadas, formam uma tabela com uma linha por grupo e uma coluna por candidato.

O erro de *k* = 25 fica abaixo do de *k* = 10 em só 6 dos 10 grupos; o erro com o logaritmo fica abaixo do erro sem ele, com o mesmo *k*, em 10 de 10. Se em quatro grupos a ordem se inverte, outro sorteio dos grupos poderia escolher outro *k*? A mesma busca, só com o logaritmo, repetida com as sementes de 0 a 4:

In [ ]:
so_k = {"modelo__n_neighbors": grade["modelo__n_neighbors"]}
escolhidos, menores = [], []
for semente in range(5):
    outros_grupos = KFold(10, shuffle=True, random_state=semente)
    outra_busca = GridSearchCV(knn, so_k, cv=outros_grupos,
                               scoring="neg_root_mean_squared_error")
    outra_busca.fit(X_treino, y_treino)
    k_semente = outra_busca.best_params_["modelo__n_neighbors"]
    escolhidos.append(k_semente)
    menores.append(-outra_busca.best_score_)
    print(f"semente {semente}: k =", k_semente,
          "| RMSE", numero_br(-outra_busca.best_score_, 4))
print("todos de 15 a 40:", all(15 <= k <= 40 for k in escolhidos))
print("RMSE de", numero_br(min(menores), 4), "a", numero_br(max(menores), 4))

O *k* escolhido muda com o sorteio dos grupos e fica, nas cinco sementes, dentro da faixa de 15 a 40 que a semente 17 mostrou. O resultado firme da busca é o logaritmo; o *k* exato é uma escolha dentro de um vale em que os vizinhos ao redor dão quase o mesmo erro, e trocar 25 por 20 quase não muda o erro.

### O que o melhor número promete

O 0,4184 é o menor de 24 números, cada um medido com o ruído de uma divisão em grupos. Com as sementes de 0 a 4, esse menor erro foi de 0,4179 a 0,4189: o mesmo treino e a mesma grade, e só o sorteio dos grupos mudou. Dois efeitos agem sobre ele, em sentidos opostos. O primeiro vem de escolher o mínimo: entre muitas medidas ruidosas, a menor tende a ser a que teve sorte no sorteio dos grupos, e cai abaixo do erro verdadeiro do candidato que a produziu. Por esse lado, o `best_score_` é otimista.

O segundo vem do tamanho de cada ajuste. Na validação cruzada, cada ajuste usa nove décimos do treino; o modelo final usa o treino inteiro e tem mais anúncios entre os quais procurar vizinhos. Por esse lado, o erro da validação cruzada tende a superestimar o do modelo final.

O saldo dos dois não se conhece de antemão, e por isso o `best_score_` não é a estimativa final do erro. Quem a dá é um conjunto que não participou da escolha, e o `teste` continua fechado.

O modelo escolhido já existe. O `GridSearchCV` reajustou o `Pipeline` com *k* = 25 e o logaritmo em todos os anúncios de treino:

In [ ]:
escolhido = busca.best_estimator_
print("anúncios no reajuste:", numero_br(escolhido["modelo"].n_samples_fit_, 0))
print("vizinhos:", escolhido["modelo"].n_neighbors)

> **🔧 Função**
>
> **`busca.best_estimator_`** — o `Pipeline` com a melhor combinação da grade, reajustado com todas as linhas de `X_treino`; é o modelo que se usa para prever.
>
> **`escolhido["modelo"].n_samples_fit_`** — quantas linhas o *k*-NN guardou no `fit`, isto é, entre quantos anúncios ele procura os vizinhos.

Os 8.267 anúncios de treino entraram no reajuste, e nenhum anúncio do `teste`. A escolha do *k* e a do logaritmo saíram da validação cruzada, dentro do treino, e é esse `Pipeline`, com o preparo e o modelo juntos, que um anúncio novo recebe.

## Reportar: a Métrica Certa, e o que o Resultado Não Diz

Quem vai anunciar um imóvel quer saber: se o modelo sugerir R$ 3.000, quanto o aluguel pedido por imóveis parecidos pode estar longe disso? Em que unidade essa resposta deve ser dada, e o que ela não autoriza a concluir?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

O ponto de partida é o da seção 17.3: a tabela sem linhas repetidas, a divisão estratificada por cidade, os dez preditores, o logaritmo do aluguel como resposta e o `preparo`.

In [ ]:
alugueis = pd.read_csv("dados/alugueis.csv", na_values=["-"]).drop_duplicates()
preditores = ["cidade", "area_m2", "quartos", "banheiros", "vagas", "andar",
              "aceita_animal", "mobiliado", "condominio", "iptu"]
treino, teste = train_test_split(alugueis, test_size=0.2, random_state=17,
                                 stratify=alugueis["cidade"])
X_treino, y_treino = treino[preditores], np.log(treino["aluguel"])

preparo = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     ["cidade", "aceita_animal", "mobiliado"]),
    ("assimetricas", Pipeline([
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("escala", StandardScaler())]), ["area_m2", "condominio", "iptu"]),
    ("andar", Pipeline([
        ("preenche", SimpleImputer(strategy="constant", fill_value=0,
                                   add_indicator=True)),
        ("escala", StandardScaler())]), ["andar"]),
    ("contagens", StandardScaler(), ["quartos", "banheiros", "vagas"]),
]).set_output(transform="pandas")
print("X_treino:", X_treino.shape)

O modelo é o da seção 17.5. O chunk abaixo refaz a busca e imprime o que ela escolheu:

In [ ]:
grupos = KFold(10, shuffle=True, random_state=17)
knn = Pipeline([("preparo", preparo), ("modelo", KNeighborsRegressor())])
grade = {
    "modelo__n_neighbors": list(range(5, 61, 5)),
    "preparo__assimetricas__log": [
        FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
        "passthrough",
    ],
}
busca = GridSearchCV(knn, grade, cv=grupos, scoring="neg_root_mean_squared_error")
busca.fit(X_treino, y_treino)
escolhido = busca.best_estimator_
com_log = not isinstance(busca.best_params_["preparo__assimetricas__log"], str)
print("k escolhido:", busca.best_params_["modelo__n_neighbors"])
print("com log:", com_log)
print("RMSE de validação cruzada:", numero_br(-busca.best_score_, 4))

### O teste, uma vez

Todas as decisões foram tomadas: as linhas repetidas, as colunas excluídas, o preparo, o modelo, o *k*. Nenhuma foi escolhida pelo erro nos 2.067 anúncios guardados. Agora eles são abertos, e o modelo escolhido prevê cada um. Antes de ver a saída: o erro no teste deve sair acima ou abaixo do erro de validação cruzada?

In [ ]:
X_teste, y_teste = teste[preditores], np.log(teste["aluguel"])
log_previsto = escolhido.predict(X_teste)
rmse_teste = root_mean_squared_error(y_teste, log_previsto)
print("anúncios de teste:", numero_br(len(teste), 0))
print("RMSE de validação cruzada:", numero_br(-busca.best_score_, 4))
print("RMSE no teste:", numero_br(rmse_teste, 4))
print("teste − validação:", numero_br(rmse_teste + busca.best_score_, 4))

> **🔧 Função**
>
> **`root_mean_squared_error(y, previsto)`** — o RMSE entre a resposta observada `y` e a `previsto`, a mesma conta do `scoring="neg_root_mean_squared_error"`, sem o sinal trocado. Vem de `sklearn.metrics`.

Os dois números ficam próximos: 0,4159 no teste e 0,4184 na validação cruzada, uma diferença de −0,0025 (0,4159 − 0,4184). O teste saiu um pouco abaixo. Isso não mede quanto o erro de validação cruzada da seção 17.5 (o `best_score_`, com o sinal trocado) era otimista ou pessimista: o teste é um sorteio só de 2.067 anúncios, e ele também oscila. Quanto? Os dez grupos da validação cruzada dão uma ideia, porque cada um é um conjunto de anúncios que o modelo não viu:

In [ ]:
i = busca.best_index_
por_grupo = pd.Series([-busca.cv_results_[f"split{g}_test_score"][i]
                       for g in range(10)])
print("grupos de validação:", len(por_grupo))
print("RMSE, menor grupo:", numero_br(por_grupo.min(), 4))
print("RMSE, maior grupo:", numero_br(por_grupo.max(), 4))
print("desvio entre grupos:", numero_br(por_grupo.std(), 4))
print("|teste − validação| < desvio:",
      bool(abs(rmse_teste + busca.best_score_) < por_grupo.std()))

> **🔧 Função**
>
> **`busca.best_index_`** — a posição do candidato escolhido nas listas de `cv_results_`; com ela, `cv_results_["split0_test_score"][i]` é o erro dele no primeiro grupo.
>
> **`pd.Series(valores, index)`** — uma coluna avulsa, com os `valores` na ordem dada; sem `index`, as posições são 0, 1, 2…

De um grupo de validação para outro, cada um com cerca de 827 anúncios (8.267 ÷ 10), o RMSE do modelo escolhido vai de 0,3969 a 0,4348, com desvio de 0,0127. A diferença entre o teste e a validação cruzada, 0,0025 em valor absoluto, é menor que esse desvio. O teste tem 2.067 anúncios, mais que os cerca de 827 de um grupo, e por isso oscila menos; mas não tanto que uma diferença desse tamanho diga para que lado está o erro verdadeiro.

A partir daqui, o teste já foi visto. Se o resultado decepcionasse e o *k* fosse trocado por causa dele, o teste teria virado um conjunto de validação a mais, e o seu número deixaria de estimar o erro num anúncio novo. O que se faz com ele agora é só descrever o erro.

### De volta aos reais

Um RMSE de 0,4159 na escala log diz pouco a quem vai anunciar. A previsão volta a reais com `np.exp`, e o erro de cada anúncio passa a ser medido em relação ao aluguel pedido:

$$
\text{erro relativo} = \frac{|\text{previsto} - \text{pedido}|}{\text{pedido}}.
$$

Um erro relativo de 0,25 é errar por 25% do aluguel pedido, para cima ou para baixo. Três medidas resumem esses erros no teste: a **mediana** do erro relativo; a **fração dos anúncios com erro de até 25%**; e o **MAE**, o erro absoluto médio, em reais. A mediana entra no lugar da média porque o erro relativo não tem teto para cima: um anúncio que pede R$ 1.000 e recebe a sugestão de R$ 3.000 erra 200%, e poucos casos assim puxam a média para cima. O limite de 25% é uma escolha de quem usa a sugestão. Com uma margem mais folgada ou mais apertada, a fração muda. Para comparar, entram a regressão linear, ajustada no mesmo treino com o mesmo preparo, e a sugestão mais ingênua possível, uma constante: a mediana dos aluguéis de treino, a mesma para todo anúncio.

In [ ]:
real = teste["aluguel"]
linear = Pipeline([("preparo", preparo), ("modelo", LinearRegression())])
linear.fit(X_treino, y_treino)
previsoes = {
    "k-NN": np.exp(log_previsto),
    "linear": np.exp(linear.predict(X_teste)),
    "constante": np.full(len(teste), treino["aluguel"].median()),
}

def resumo(previsto):
    relativo = np.abs(previsto - real) / real
    return {"mediano (%)": 100 * relativo.median(),
            "até 25% (%)": 100 * (relativo <= 0.25).mean(),
            "MAE (R$)": mean_absolute_error(real, previsto),
            "RMSE (R$)": root_mean_squared_error(real, previsto)}

metricas = pd.DataFrame({nome: resumo(p) for nome, p in previsoes.items()}).T
erro_knn = np.abs(previsoes["k-NN"] - real)
print("k-NN, erro relativo médio:",
      numero_br(100 * (erro_knn / real).mean(), 1) + "%")
print("k-NN, erro relativo mediano:",
      numero_br(100 * (erro_knn / real).median(), 1) + "%")
print("k-NN, erro mediano: R$", numero_br(erro_knn.median(), 0))
(metricas.style
 .format(lambda v: numero_br(v, 1), subset=["mediano (%)", "até 25% (%)"])
 .format(lambda v: numero_br(v, 0), subset=["MAE (R$)", "RMSE (R$)"]))

> **🔧 Função**
>
> **`np.abs(x)`** — o valor absoluto, elemento a elemento.
>
> **`mean_absolute_error(y, previsto)`** — o MAE: a média de $|y - \text{previsto}|$, na unidade de `y`, aqui em reais. Vem de `sklearn.metrics`.

A coluna "mediano" é a mediana do erro relativo; "até 25%" é a fração dos anúncios com erro relativo de até 25%. No anúncio típico do teste, o *k*-NN erra 26,8% do aluguel pedido, e 46,7% dos anúncios têm erro de até 25%. A linear erra 30,0% no anúncio típico e a constante, 54,2%. O erro relativo médio do *k*-NN, 33,8%, fica acima do mediano, 26,8%: os anúncios com erro grande puxam a média, e é por isso que o relato usa a mediana. Em reais, o *k*-NN erra em média R$ 1.391 por anúncio, contra R$ 1.470 da linear e R$ 2.374 da constante. O MAE também é uma média, e os erros em reais dos aluguéis altos o puxam: o erro mediano do *k*-NN é de R$ 693, cerca de metade do MAE.

A tabela traz também o RMSE em reais, que o resumo acima deixou de lado. O quadrado dá peso desproporcional aos erros grandes, e os erros grandes em reais estão nos aluguéis altos. Quanto pesam os poucos piores anúncios na soma dos quadrados?

In [ ]:
quadrados = (previsoes["k-NN"] - real) ** 2
piores = quadrados.nlargest(round(0.01 * len(quadrados))).index
print("1% dos anúncios:", len(piores))
print("fatia da soma dos quadrados:",
      numero_br(100 * quadrados[piores].sum() / quadrados.sum(), 1) + "%")
print("só o pior:",
      numero_br(100 * quadrados.max() / quadrados.sum(), 1) + "%")
metade = quadrados.nsmallest(len(quadrados) // 2)
print("metade de menor erro:", numero_br(len(metade), 0))
print("fatia dessa metade:",
      numero_br(100 * metade.sum() / quadrados.sum(), 1) + "%")
print("aluguel mediano deles: R$", numero_br(real[piores].median(), 0))
print("aluguel mediano, teste: R$", numero_br(real.median(), 0))
print("teste com R$ 15.000:", (real == 15_000).sum())
print("teste acima de R$ 15.000:", (real > 15_000).sum())
print("piores com R$ 15.000:", (real[piores] == 15_000).sum())
no_teto = (real == 15_000).to_numpy()
print("previstos abaixo, todos:",
      bool((previsoes["k-NN"][no_teto] < 15_000).all()))

> **🔧 Função**
>
> **`serie.nsmallest(n)`** — os `n` menores valores da série, do menor para o maior, com os rótulos das linhas; é o espelho de `nlargest`.

Os 21 piores anúncios, 1% do teste, respondem por 29,7% da soma dos quadrados, e o aluguel mediano deles é de R$ 15.000, contra R$ 2.740 no teste inteiro. O pior de todos, sozinho, responde por 9,6%; os 1.033 anúncios de menor erro, metade do teste, somam 1,1%. Corrigir a previsão daquele um mexeria no RMSE mais do que acertar em cheio a metade inteira. Dos 21 piores, 12 pedem exatamente R$ 15.000, e o valor chama atenção por si: 54 anúncios de teste pedem exatamente isso, e só 5 pedem mais. Pode ser um teto aplicado a parte da coleta, com aluguéis maiores registrados como R$ 15.000; os 5 acima dele mostram que, se houve teto, ele não valeu para todos, e o arquivo não diz. O RMSE em reais é decidido, em boa parte, por um punhado de imóveis caros, e por isso não serve como número principal.

Falta uma pergunta sobre a volta a reais. O modelo foi ajustado para acertar o logaritmo do aluguel, e $e^{\text{média dos logaritmos}}$ não é a média dos aluguéis. A exponencial estica mais os valores altos que os baixos, e por isso $e^{\text{média dos logaritmos}}$ não fica acima da média (é a desigualdade de Jensen); ela fica perto da mediana quando o logaritmo tem distribuição quase simétrica. A previsão herda isso?

In [ ]:
previsto = pd.Series(previsoes["k-NN"], index=teste.index)
print("média prevista: R$", numero_br(previsto.mean(), 0))
print("média pedida: R$", numero_br(real.mean(), 0))
print("mediana prevista: R$", numero_br(previsto.median(), 0))
print("mediana pedida: R$", numero_br(real.median(), 0))

As medianas ficam próximas, R$ 2.688 previstos contra R$ 2.740 pedidos; a média prevista, R$ 3.466, fica abaixo da média pedida, R$ 3.993. Para um anúncio, a sugestão é um preço típico, perto do meio dos anúncios parecidos. Somadas para muitos anúncios, as previsões subestimam o total. Quem quiser estimar a receita de uma carteira de imóveis precisa de outro alvo ou de uma correção que este modelo não tem.

> **🔷 Conceito**
>
> O número que se reporta é escolhido pela pergunta de quem vai usá-lo, não pelo que o modelo otimizou. Para "quanto pedir por este anúncio", o relato é o **erro relativo mediano**, 26,8%, e a **fração dos anúncios com erro de até 25%**, 46,7%; o **MAE**, R$ 1.391, traduz o erro para reais. O RMSE na escala log serviu para escolher o modelo; o RMSE em reais fica de fora, porque um punhado de imóveis caros o decide.

A figura abaixo mostra os 2.067 anúncios de teste, cada um com o aluguel pedido e o previsto pelo *k*-NN:

In [ ]:
# Figura: Aluguel previsto pelo k-NN contra o aluguel pedido, nos 2.067 anúncios de teste, com os dois eixos em escala logarítmica. A diagonal cinza marca a previsão exata; a margem cinza em volta dela, os anúncios com erro de até 25% do aluguel pedido, que são 46,7% do teste.
marcas = [500, 1_000, 2_000, 5_000, 10_000, 20_000, 40_000]
marcas_x = [500, 1_000, 3_000, 10_000, 30_000]
linha = np.geomspace(400, 50_000, 100)
fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.fill_between(linha, 0.75 * linha, 1.25 * linha, color="0.45", alpha=0.2,
                linewidth=0)
ax.plot(linha, linha, color="0.45", linewidth=1.5)
ax.scatter(real, previsto, s=8, color="C0", alpha=0.3, linewidths=0)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(marcas_x, [numero_br(m, 0) for m in marcas_x])
ax.set_yticks(marcas, [numero_br(m, 0) for m in marcas])
ax.set_xlim(400, 50_000)
ax.set_ylim(400, 50_000)
ax.set_xlabel("aluguel pedido (R$)")
ax.set_ylabel("aluguel previsto (R$)")
plt.tight_layout()
plt.show()

Em reais, a margem é mais larga nos aluguéis altos, porque 25% de um aluguel alto são mais reais que 25% de um baixo; nos eixos em log, ela tem a mesma largura no gráfico inteiro. A coluna vertical de pontos em R$ 15.000 é formada pelos 54 anúncios desse valor, todos previstos abaixo dele. A nuvem parece inclinada em relação à diagonal, mais achatada que ela. O modelo subestima os imóveis caros?

> **🔧 Função**
>
> **`ax.fill_between(x, y_baixo, y_alto, color, alpha, linewidth)`** — pinta a região entre as curvas `y_baixo` e `y_alto` ao longo de `x`. Aqui, entre 0,75 e 1,25 vezes o aluguel: a margem de ±25%.

### Onde erra mais

O erro relativo mediano de 26,8% resume o teste inteiro, e o erro não se distribui igualmente. Primeiro, cidade por cidade:

In [ ]:
relativo = (previsto - real).abs() / real
por_cidade = relativo.groupby(teste["cidade"]).agg(["median", "size"])
por_cidade["median"] = 100 * por_cidade["median"]
(por_cidade.sort_values("median")
 .rename(columns={"median": "mediano (%)", "size": "anúncios"})
 .rename_axis(None)
 .style.format(lambda v: numero_br(v, 1), subset=["mediano (%)"])
 .format(lambda v: numero_br(v, 0), subset=["anúncios"]))

> **🔧 Função**
>
> **`serie.abs()`** — o valor absoluto, elemento a elemento, como `np.abs`.
>
> **`serie.groupby(chave).agg(funcoes)`** — separa a série pelos valores de `chave` e aplica cada função da lista a cada grupo; `"median"` é a mediana e `"size"`, o número de linhas do grupo. Quando a `chave` é categórica, como as faixas do `pd.qcut` adiante, `groupby(chave, observed=True)` fica só com as categorias que aparecem nos dados.
>
> **`df.sort_values(coluna)`** — as linhas em ordem crescente de `coluna`.
>
> **`df.rename(columns=dicionario)`** — troca os nomes das colunas pelos do dicionário `{antigo: novo}`.

Porto Alegre tem o menor erro relativo mediano, 21,2%, e São Paulo o maior, 29,4%. As medianas não têm a mesma firmeza: a de São Paulo sai de 1.143 anúncios, e a de Campinas, de 165. Quanto menor o grupo, mais a mediana dele depende de quais anúncios caíram no teste, e parte da diferença entre cidades pode ser acaso da divisão.

A figura sugere uma segunda divisão, pelo preço. Mas o eixo horizontal dela é o aluguel pedido, e cortar por ele engana. Uma previsão feita só com as características, por melhor que seja, puxa para o meio: entre os anúncios que pedem mais estão justamente os que pedem mais do que as suas características indicam, e a previsão fica abaixo deles; entre os que pedem menos, acontece o contrário. É a regressão à média, e ela achata a nuvem mesmo que, para cada tipo de imóvel, a previsão acerte o pedido típico. Quem vai anunciar conhece a previsão, e é por ela que se corta. O `pd.qcut` divide os anúncios de teste em cinco faixas de aluguel **previsto**, com o mesmo número de anúncios cada, e o chunk mede, em cada faixa, o erro relativo mediano e o erro com sinal: positivo quando a previsão fica acima do aluguel pedido, negativo quando fica abaixo.

In [ ]:
faixa = pd.qcut(previsto, 5)
com_sinal = (previsto - real) / real
por_faixa = pd.DataFrame({
    "mediano (%)": 100 * relativo.groupby(faixa, observed=True).median(),
    "com sinal (%)": 100 * com_sinal.groupby(faixa, observed=True).median(),
})
por_faixa.index = [f"{numero_br(f.left, 0)} a {numero_br(f.right, 0)}"
                   for f in por_faixa.index]
print("maior erro:", por_faixa["mediano (%)"].idxmax())
print("menor erro:", por_faixa["mediano (%)"].idxmin())
print("cresce faixa a faixa:",
      por_faixa["mediano (%)"].is_monotonic_increasing)
por_faixa.style.format(lambda v: numero_br(v, 1))

> **🔧 Função**
>
> **`pd.qcut(serie, q)`** — corta os valores em `q` faixas com o mesmo número de linhas, pelos quantis; cada valor recebe o intervalo a que pertence, com as bordas em `.left` e `.right`.
>
> **`serie.idxmin()`** — o rótulo da linha em que está o menor valor.
>
> **`serie.is_monotonic_increasing`** — `True` se cada valor é maior ou igual ao anterior.

O erro relativo mediano cresce com o preço previsto, faixa a faixa: cerca de 20% na mais barata, de R$ 692 a R$ 1.510, e 34,3% na mais cara, de R$ 5.606 a R$ 12.035. O viés quase não aparece nas quatro primeiras faixas, com erro com sinal de 1,6%, −0,6%, 1,8% e −2,8%. Na faixa de cima, a previsão fica, no anúncio típico, 11,5% abaixo do aluguel pedido: é ali que o modelo subestima. Para o dono de um imóvel com previsão alta, a sugestão é menos precisa que a típica e tende a sair abaixo do que anúncios como o dele pedem.

Quanto da inclinação da nuvem vem do corte? A figura abaixo mede o erro com sinal nas cinco faixas de aluguel previsto, as da tabela, e nas cinco faixas de aluguel pedido, com os mesmos anúncios e o mesmo modelo:

In [ ]:
# Figura: Erro com sinal mediano do k-NN nas cinco faixas de aluguel, da mais barata à mais cara, com os 2.067 anúncios de teste cortados de dois jeitos: pelo aluguel previsto (azul) e pelo aluguel pedido (laranja). Acima de zero, a previsão fica acima do pedido.
faixa_pedido = pd.qcut(real, 5)
por_pedido = 100 * com_sinal.groupby(faixa_pedido, observed=True).median()
print("pelo pedido, mais barata:", numero_br(por_pedido.iloc[0], 1) + "%")
print("pelo pedido, mais cara:", numero_br(por_pedido.iloc[-1], 1) + "%")
print("amplitude, pelo pedido:",
      numero_br(por_pedido.max() - por_pedido.min(), 1))
print("amplitude, pelo previsto:",
      numero_br(por_faixa["com sinal (%)"].max()
                - por_faixa["com sinal (%)"].min(), 1))
posicoes = range(1, 6)
fig, ax = plt.subplots(figsize=(4.6, 3.4))
ax.axhline(0, color="0.45", linewidth=1)
ax.plot(posicoes, por_faixa["com sinal (%)"], color="C0", marker="o",
        label="faixas do previsto")
ax.plot(posicoes, por_pedido, color="C1", marker="o",
        label="faixas do pedido")
ax.set_xticks(posicoes, ["1ª", "2ª", "3ª", "4ª", "5ª"])
ax.set_xlim(0.5, 5.5)
ax.set_xlabel("faixa de aluguel, da mais barata à mais cara")
ax.set_ylabel("erro com sinal mediano (%)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.axhline(y, color, linewidth)`** — traça uma linha horizontal na altura `y`, de uma borda à outra do gráfico. Aqui, no zero, que separa as previsões acima do pedido das previsões abaixo dele.

Cortado pelo aluguel pedido, o erro com sinal vai de 23,5% na faixa mais barata a −34,3% na mais cara: o modelo pareceria superestimar os baratos e subestimar muito os caros. Cortado pelo previsto, fica perto de zero até a quarta faixa e chega a −11,5% na quinta. Da faixa de erro com sinal mais alto à de mais baixo, a distância é de 57,9 pontos percentuais no corte pelo pedido e de 13,2 no corte pelo previsto. A diferença, 44,7 pontos (57,9 − 13,2), é a maior parte da inclinação da nuvem, e vem da regressão à média, criada pelo corte; o que sobra, a subestimação na faixa de cima, é do modelo.

Resta a pergunta do começo, em reais. O chunk abaixo toma os anúncios de teste previstos entre R$ 2.500 e R$ 3.500 e mede, em cada um, a razão entre o aluguel pedido e o previsto; os quartis dessa razão, aplicados a R$ 3.000, dão o intervalo em que cai a metade central dos pedidos:

In [ ]:
perto = previsto.between(2_500, 3_500)
q1, q3 = (real[perto] / previsto[perto]).quantile([0.25, 0.75])
print("previstos de 2.500 a 3.500:", perto.sum())
print("razão, quartil de baixo:", numero_br(q1, 2))
print("razão, quartil de cima:", numero_br(q3, 2))
print("quartil de baixo: R$", numero_br(3_000 * q1, 0))
print("quartil de cima: R$", numero_br(3_000 * q3, 0))

> **🔧 Função**
>
> **`serie.between(a, b)`** — `True` onde o valor está entre `a` e `b`, com as duas pontas incluídas.

Nos 319 anúncios previstos perto de R$ 3.000, um em cada quatro pede menos que 0,74 vez a previsão, e um em cada quatro, mais que 1,31 vez. Aplicadas a uma sugestão de R$ 3.000, essas razões (com as razões sem arredondar) põem a metade central dos pedidos parecidos entre R$ 2.224 e R$ 3.927. É essa a resposta a quem vai anunciar: a sugestão aponta o meio do mercado, e a margem em volta dela vai de R$ 776 para baixo (3.000 − 2.224) a R$ 927 para cima (3.927 − 3.000).

### O que o resultado não diz

Os números acima medem quão perto o modelo chega do aluguel **pedido** em anúncios que ele não viu. Várias perguntas próximas ficam sem resposta.

**O aluguel é o pedido, não o contratado.** O arquivo registra o valor do anúncio. Se os imóveis caros costumam fechar abaixo do anunciado, ou se os anúncios mais baratos saem do ar mais depressa, nada disso está na tabela, e o modelo aprende a pedir como os anunciantes pedem.

**Cinco cidades, numa data que o arquivo não registra.** Os anúncios de teste vêm das cinco cidades da tabela acima, na mesma proporção do treino, porque a divisão foi estratificada por cidade. Um anúncio de outra cidade cai no `handle_unknown="ignore"` da seção 17.3, com as cinco colunas de cidade zeradas, e o erro medido aqui não vale para ele. E o arquivo não tem a data da coleta: o erro vale para o mercado daquele momento, e aluguéis mudam com o tempo.

**Associação não é efeito.** O modelo usa `mobiliado`, como as outras colunas, para achar anúncios parecidos. Nada no resultado diz quanto o aluguel de um imóvel subiria se o dono o mobiliasse: os imóveis mobiliados podem diferir dos outros em localização, padrão de acabamento ou público, e nada disso está nas colunas. Um *k*-NN nem sequer tem um coeficiente para `mobiliado`, e mesmo o coeficiente de uma regressão linear só descreveria como mobília e aluguel andam juntos nestes anúncios.

**A regra das linhas repetidas é uma suposição.** A tabela perdeu as linhas repetidas antes da divisão, supondo que cada repetição é o mesmo anúncio publicado de novo. A seção 6.4 mostrou anúncios idênticos até o último real, e sem uma coluna que identifique o anúncio não há como saber se são um imóvel ou dois. Se forem dois, a tabela perdeu anúncios legítimos. Foi uma escolha, e o erro do teste foi medido depois dela.

**O teste só vale uma vez.** O RMSE de 0,4159 e o erro relativo mediano de 26,8% estimam o erro num anúncio novo porque nenhuma decisão foi tomada pelo erro no teste. Voltar a mexer no modelo depois de ver esses números (trocar o *k*, tirar uma coluna, experimentar outro modelo e ficar com o que foi melhor no teste) faz do teste um conjunto de validação, e aí um número honesto exige outro conjunto de anúncios, que nenhuma decisão tenha visto.

## Leituras adicionais

- O capítulo 2 de Géron (2022), um projeto de ponta a ponta sobre preços de imóveis na Califórnia, com a separação do teste, o `ColumnTransformer`, o `Pipeline` e o `GridSearchCV`.
- O capítulo 4 de Bruce et al. (2020), com a regressão aplicada a preços de venda de imóveis e a codificação de variáveis categóricas.
- [O guia "Common pitfalls and recommended practices" do scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html), sobre pré-processamento inconsistente entre treino e teste, vazamento de dados e sementes.
- [O guia de composição de estimadores do scikit-learn](https://scikit-learn.org/stable/modules/compose.html), com o `Pipeline`, o `ColumnTransformer` e o acesso aos parâmetros aninhados com o duplo sublinhado (`__`).

## Referências

- **Bruce; Bruce; Gedeck**. *Practical Statistics for Data Scientists*. 2nd ed. O'Reilly Media. 2020.
- **Géron**. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 3rd ed. O'Reilly Media. 2022.